In [ ]:
# import os
# for root, dirs, files in os.walk('/content/drive/MyDrive/RBC Capstone'):
#     for f in files:
#         if f.endswith('.parquet'):
#             print(os.path.join(root, f))

In [ ]:
import os

# Walk from MyDrive and find the Input folder
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for d in dirs:
        if 'input' in d.lower() or 'capstone' in d.lower():
            print(os.path.join(root, d))
    for f in files:
        if f.endswith('.csv') and 'party' in f.lower():
            print(os.path.join(root, f))

/content/drive/MyDrive/RBC Capstone 
/content/drive/MyDrive/RBC Capstone
/content/drive/MyDrive/RBC Capstone /Input
/content/drive/MyDrive/RBC Capstone /Input/Party.csv
/content/drive/MyDrive/RBC Capstone /Input/PartyRelationship.csv
/content/drive/MyDrive/RBC Capstone /Input/PartyServiceDetails.csv
/content/drive/MyDrive/RBC Capstone /Input/HouseholdPartyRelationship.csv
/content/drive/MyDrive/RBC Capstone /Input/PartyWMOandAggregationDetails.csv


In [1]:
# # Cell 1
# !pip install xgboost --quiet

# # Cell 2
# from google.colab import drive
# drive.mount('/content/drive')

# # Cell 3 — then set INPUT_PATH in the script to match your Drive folder
# # e.g. '/content/drive/MyDrive/Capstone Data'

# # Cell 1
# !pip install xgboost --quiet

# # Cell 2 — upload your parquet file directly
# from google.colab import files
# uploaded = files.upload()  # a file picker will pop up — select master_modeling_table.parquet

# Cell 1
!pip install xgboost --quiet

# Cell 2
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# RBC BENEFICIARY CONVERSION MODEL — FINAL PIPELINE (LEAKAGE-FIXED)
# ══════════════════════════════════════════════════════════════════════════════
#
# LEAKAGE FIXES APPLIED:
#
#   1. CAMPAIGN FEATURES REMOVED from MODEL_FEATURES:
#      n_campaigns_targeted, ever_responded, ever_opted_out_email, ever_do_not_call
#      -> These columns come from CampaignMember which only contains EXISTING CLIENTS.
#         Non-clients are never in campaign records, so the model was learning
#         "has campaign record = is client" — perfect circular leakage.
#         100% of non-clients had zero for all campaign features.
#      -> Still computed and kept in master table for OUTPUT ENRICHMENT only.
#
#   2. WMO / DIGITAL FEATURES REMOVED from MODEL_FEATURES:
#      registered_in_wmo, has_aggregated_accounts
#      -> WMO registration only happens AFTER becoming a client.
#         100% zero for non-clients — pure leakage.
#      -> Still computed and kept for output enrichment.
#
#   3. RELATIONSHIP NETWORK FEATURES REMOVED from MODEL_FEATURES:
#      n_client_relationships, n_total_relationships, pct_relations_are_clients
#      -> PartyRelationship table is only populated for existing clients.
#         All non-clients had 0 for every relationship feature — pure leakage.
#      -> Still computed and kept for output enrichment.
#
#   4. MARITAL STATUS DUMMIES REMOVED from MODEL_FEATURES:
#      All marital_* columns
#      -> Marital status is only recorded for clients in this dataset.
#         Non-clients show near-100% zero rates (marital_Single is 74.6% for
#         non-clients vs 21.4% for clients; marital_Domestic Partner is 100%
#         zero for non-clients). These columns effectively encode is_client.
#      -> marital_* kept in master table for inspection but not used in model.
#
#   5. HOUSEHOLD FEATURES — RETAINED but documented:
#      in_client_household, n_households
#      -> These show ratio ~32-35x. A non-client can legitimately appear in a
#         client household (e.g. a spouse or child), which is a genuine
#         predictive signal. Retained but flagged.
#      -> If AUC is still > 0.90 after these fixes, also remove these two
#         features and re-run.
#
# COLAB SETUP — run these cells BEFORE running this script:
#
#   Cell 1:
#   !pip install xgboost --quiet
#
#   Cell 2:
#   from google.colab import drive
#   drive.mount('/content/drive')
#
# All CSV files must be inside:
#   /content/drive/MyDrive/RBC Capstone /Input/
#
# Outputs will be saved to:
#   /content/drive/MyDrive/RBC Capstone /Model Output/
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os, glob, warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, roc_curve,
                              precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════════
DATA_PATH   = '/content/drive/MyDrive/RBC Capstone /Input'
OUTPUT_PATH = '/content/drive/MyDrive/RBC Capstone /Model Output'
os.makedirs(OUTPUT_PATH, exist_ok=True)

all_files = os.listdir(DATA_PATH)
print(f"Files found in Input folder: {len(all_files)}")
hist_files = sorted([f for f in all_files
                     if 'BroadridgeAccountAdditionalHistory' in f
                     and f.endswith('.csv')])
print(f"History files found: {len(hist_files)}")
for f in hist_files:
    print(f"  {f}")

# ══════════════════════════════════════════════════════════════════════════════
# UTILITY
# ══════════════════════════════════════════════════════════════════════════════
def read_csv(fname, **kw):
    path = os.path.join(DATA_PATH, fname)
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not read: {path}")

def str_col(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD SOURCE FILES
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading source files...")

party   = read_csv('Party.csv')
acct    = read_csv('BroadridgeAccount.csv')
role    = read_csv('BroadridgeAccountRole.csv')
yodlee  = read_csv('YodleeAccount.csv')
hpr     = read_csv('HouseholdPartyRelationship.csv')
hh      = read_csv('Household.csv')
pr      = read_csv('PartyRelationship.csv')
psd     = read_csv('PartyServiceDetails.csv')
wmo     = read_csv('PartyWMOandAggregationDetails.csv')
cm      = read_csv('CampaignMember.csv')
contrib = read_csv('BroadridgeAccountContributionHistory.csv')
dist    = read_csv('BroadridgeAccountDistributionHistory.csv')

print("Loading history files (may take ~2 mins on Drive)...")
hist_dfs = []
for f in hist_files:
    print(f"  Reading {f}...")
    hist_dfs.append(read_csv(f))
hist = pd.concat(hist_dfs, ignore_index=True)
print(f"  Total history rows: {len(hist):,}")

# Normalise ID columns
for df, cols in [
    (party,   ['PartyId']),
    (acct,    ['PrimaryOwnerPartyId', 'FinancialAccountId']),
    (role,    ['PartyId', 'AccountId']),
    (yodlee,  ['PrimaryOwnerPartyId']),
    (hpr,     ['PartyId', 'HouseholdId']),
    (pr,      ['PartyId', 'RelatedPartyId']),
    (psd,     ['PartyId']),
    (wmo,     ['PartyId']),
    (cm,      ['PartyId']),
    (contrib, ['FinancialAccountId']),
    (dist,    ['FinancialAccountId']),
    (hist,    ['FinancialAccountId']),
]:
    for c in cols:
        str_col(df, c)

print(f"\nParties:      {len(party):,}")
print(f"Accounts:     {len(acct):,}")
print(f"Roles:        {len(role):,}")
print(f"Non-clients:  {(party['ClientIndicator']=='N').sum():,}")
print(f"Clients:      {(party['ClientIndicator']=='Y').sum():,}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — ACCOUNT-LEVEL FEATURE AGGREGATION
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding account-level features...")

SNAPSHOT = pd.Timestamp('2026-02-27')

hist['DeltaDate'] = pd.to_datetime(hist['DeltaDate'], errors='coerce')
for c in ['PortfolioValue','EquityPercentage','MarginBalance',
          'CashBalance','FederalCall','HouseCall','YTDMarginInterest']:
    hist[c] = pd.to_numeric(hist[c], errors='coerce').fillna(0)

hist_latest = (hist.sort_values('DeltaDate')
                   .groupby('FinancialAccountId')
                   .agg(
                       acct_portfolio_value     = ('PortfolioValue',    'last'),
                       acct_equity_pct          = ('EquityPercentage',  'last'),
                       acct_margin_balance      = ('MarginBalance',     'last'),
                       acct_cash_balance        = ('CashBalance',       'last'),
                       acct_federal_call        = ('FederalCall',       'max'),
                       acct_house_call          = ('HouseCall',         'max'),
                       acct_ytd_margin          = ('YTDMarginInterest', 'last'),
                       acct_peak_portfolio      = ('PortfolioValue',    'max'),
                       acct_min_portfolio       = ('PortfolioValue',    'min'),
                       acct_n_history_snapshots = ('DeltaDate',         'count'),
                   ).reset_index())

for c in ['CYTDDirectRollover','CYTDIRAContribution','CYTDEmployeeContribution']:
    contrib[c] = pd.to_numeric(contrib[c], errors='coerce').fillna(0)

contrib_agg = contrib.groupby('FinancialAccountId').agg(
    acct_cytd_rollover    = ('CYTDDirectRollover',      'sum'),
    acct_cytd_ira_contrib = ('CYTDIRAContribution',      'sum'),
    acct_cytd_emp_contrib = ('CYTDEmployeeContribution', 'sum'),
    acct_has_rollover     = ('CYTDDirectRollover', lambda x: int((x > 0).any())),
).reset_index()

for c in ['CYTDGrossDistribution','PYTDGrossDistribution']:
    dist[c] = pd.to_numeric(dist[c], errors='coerce').fillna(0)

dist_agg = dist.groupby('FinancialAccountId').agg(
    acct_cytd_distribution = ('CYTDGrossDistribution', 'sum'),
    acct_pytd_distribution = ('PYTDGrossDistribution', 'sum'),
    acct_has_distribution  = ('CYTDGrossDistribution', lambda x: int((x > 0).any())),
).reset_index()

acct['OpenDate'] = pd.to_datetime(acct['OpenDate'], errors='coerce')
acct['acct_age_years']   = ((SNAPSHOT - acct['OpenDate']).dt.days / 365.25).clip(lower=0)
acct['acct_is_advisory'] = (acct['AdvisoryAccount'].astype(str).str.lower() == 'true').astype(int)
acct['acct_has_rmd']     = acct['RequiredMinimumDistribution'].notna().astype(int)
acct['acct_is_ira']      = acct['AccountSubType'].str.contains(
    'IRA|Roth|Retirement|Sep|Simple|Sarsep|Rollover', case=False, na=False).astype(int)
acct['acct_is_trust']    = acct['AccountSubType'].str.contains(
    'Trust|Grantor|Irrevocable', case=False, na=False).astype(int)
acct['acct_is_joint']    = acct['AccountSubType'].str.contains(
    'Joint|WROS|Tenants', case=False, na=False).astype(int)
acct['acct_is_529']      = acct['AccountSubType'].str.contains(
    '529', case=False, na=False).astype(int)
acct['acct_is_open']     = (acct['AccountStatus'] == 'Account Open').astype(int)

owner_age = (party[['PartyId','PartyAge']]
             .rename(columns={'PartyId':'PrimaryOwnerPartyId', 'PartyAge':'primary_owner_age'}))
owner_age['primary_owner_age'] = pd.to_numeric(owner_age['primary_owner_age'], errors='coerce')

acct_feat = (acct
    .merge(hist_latest, on='FinancialAccountId', how='left')
    .merge(contrib_agg, on='FinancialAccountId', how='left')
    .merge(dist_agg,    on='FinancialAccountId', how='left')
    .merge(owner_age,   on='PrimaryOwnerPartyId', how='left')
)
num_acct = acct_feat.select_dtypes(include=[np.number]).columns
acct_feat[num_acct] = acct_feat[num_acct].fillna(0)
acct_feat['acct_primary_owner_70plus'] = (acct_feat['primary_owner_age'] >= 70).astype(int)
acct_feat['acct_primary_owner_80plus'] = (acct_feat['primary_owner_age'] >= 80).astype(int)

print(f"  Account features: {len(acct_feat):,} accounts")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — NON-CLIENT FEATURES FROM ASSOCIATED ACCOUNTS
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding non-client features from associated accounts...")

ACCT_COLS = [
    'FinancialAccountId',
    'acct_portfolio_value', 'acct_peak_portfolio', 'acct_min_portfolio',
    'acct_equity_pct', 'acct_cash_balance', 'acct_margin_balance',
    'acct_ytd_margin', 'acct_federal_call', 'acct_house_call',
    'acct_cytd_rollover', 'acct_cytd_ira_contrib', 'acct_cytd_emp_contrib',
    'acct_cytd_distribution', 'acct_pytd_distribution',
    'acct_has_rollover', 'acct_has_distribution',
    'acct_age_years', 'acct_is_advisory', 'acct_has_rmd',
    'acct_is_ira', 'acct_is_trust', 'acct_is_joint', 'acct_is_529',
    'acct_is_open', 'acct_n_history_snapshots',
    'primary_owner_age', 'acct_primary_owner_70plus', 'acct_primary_owner_80plus',
]

role_acct = role.merge(acct_feat[ACCT_COLS],
                       left_on='AccountId', right_on='FinancialAccountId', how='left')
role_acct = role_acct[role_acct['acct_is_open'].fillna(0) == 1].copy()

role_acct['r_primary_bene']    = (role_acct['FinancialAccountRole'] == 'Primary Beneficiary').astype(int)
role_acct['r_contingent_bene'] = (role_acct['FinancialAccountRole'] == 'Contingent Beneficiary').astype(int)
role_acct['r_tod_primary']     = (role_acct['FinancialAccountRole'] == 'Transfer on Death (TOD) Bene Primary').astype(int)
role_acct['r_tod_contingent']  = (role_acct['FinancialAccountRole'] == 'Transfer on Death (TOD) Bene Contingent').astype(int)
role_acct['r_designated_bene'] = (role_acct['FinancialAccountRole'] == 'Designated Beneficiary').astype(int)
role_acct['r_trusted_contact'] = (role_acct['FinancialAccountRole'] == 'Trusted Contact').astype(int)
role_acct['r_minor']           = (role_acct['FinancialAccountRole'] == 'Minor').astype(int)
role_acct['r_any_bene']        = (role_acct[['r_primary_bene','r_contingent_bene',
                                              'r_tod_primary','r_tod_contingent',
                                              'r_designated_bene']].sum(axis=1) > 0).astype(int)

nc_roles = role_acct.groupby('PartyId').agg(
    n_accounts_named_on          = ('AccountId',                'nunique'),
    n_as_primary_bene            = ('r_primary_bene',           'sum'),
    n_as_contingent_bene         = ('r_contingent_bene',        'sum'),
    n_as_tod_bene                = ('r_tod_primary',            'sum'),
    n_as_trusted_contact         = ('r_trusted_contact',        'sum'),
    n_as_minor                   = ('r_minor',                  'sum'),
    n_as_any_bene                = ('r_any_bene',               'sum'),
    associated_portfolio_total   = ('acct_portfolio_value',     'sum'),
    associated_portfolio_max     = ('acct_portfolio_value',     'max'),
    associated_portfolio_mean    = ('acct_portfolio_value',     'mean'),
    associated_peak_portfolio    = ('acct_peak_portfolio',      'max'),
    n_advisory_accts             = ('acct_is_advisory',         'sum'),
    n_ira_accts                  = ('acct_is_ira',              'sum'),
    n_trust_accts                = ('acct_is_trust',            'sum'),
    n_joint_accts                = ('acct_is_joint',            'sum'),
    n_529_accts                  = ('acct_is_529',              'sum'),
    associated_acct_age_max      = ('acct_age_years',           'max'),
    associated_acct_age_mean     = ('acct_age_years',           'mean'),
    any_owner_70plus             = ('acct_primary_owner_70plus','max'),
    any_owner_80plus             = ('acct_primary_owner_80plus','max'),
    max_owner_age                = ('primary_owner_age',        'max'),
    associated_rollover_total    = ('acct_cytd_rollover',       'sum'),
    associated_distribution_total= ('acct_cytd_distribution',   'sum'),
    associated_has_rollover      = ('acct_has_rollover',        'max'),
    associated_has_distribution  = ('acct_has_distribution',    'max'),
    associated_has_rmd           = ('acct_has_rmd',             'max'),
    associated_margin_balance    = ('acct_margin_balance',      'sum'),
    associated_any_federal_call  = ('acct_federal_call',        'max'),
    associated_any_house_call    = ('acct_house_call',          'max'),
    associated_n_snapshots       = ('acct_n_history_snapshots', 'sum'),
).reset_index()

nc_roles['has_advisory_associated'] = (nc_roles['n_advisory_accts'] > 0).astype(int)
nc_roles['has_ira_associated']      = (nc_roles['n_ira_accts'] > 0).astype(int)
nc_roles['has_trust_associated']    = (nc_roles['n_trust_accts'] > 0).astype(int)
nc_roles['multi_account_named']     = (nc_roles['n_accounts_named_on'] > 1).astype(int)
nc_roles['log_portfolio']           = np.log1p(nc_roles['associated_portfolio_total'])
nc_roles['log_max_portfolio']       = np.log1p(nc_roles['associated_portfolio_max'])

print(f"  Non-client role features: {len(nc_roles):,} parties")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — HOUSEHOLD FEATURES
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding household features...")

hh['is_client_hh'] = (hh['HouseholdType'] == 'Client Household').astype(int)
party_hh = hpr.merge(hh[['HouseholdId','is_client_hh']], on='HouseholdId', how='left')
party_hh['is_client_hh'] = party_hh['is_client_hh'].fillna(0)

hh_feats = party_hh.groupby('PartyId').agg(
    in_client_household = ('is_client_hh', 'max'),
    n_households        = ('HouseholdId',  'nunique'),
).reset_index()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — CAMPAIGN & WMO FEATURES
# ENRICHMENT ONLY — excluded from MODEL_FEATURES (see header for reason)
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding campaign/WMO features (enrichment only — NOT in model)...")

cm_feats = cm.groupby('PartyId').agg(
    n_campaigns_targeted = ('CampaignId',        'nunique'),
    ever_responded       = ('HasResponded',       'max'),
    ever_opted_out_email = ('HasOptedOutOfEmail', 'max'),
    ever_do_not_call     = ('DoNotCall',          'max'),
).reset_index()
for c in ['ever_responded','ever_opted_out_email','ever_do_not_call']:
    cm_feats[c] = cm_feats[c].astype(int)

wmo['registered_in_wmo']       = (wmo['PartyRegisteredInWMO'].astype(str).str.lower() == 'true').astype(int)
wmo['has_aggregated_accounts'] = (wmo['PartyAggregatedAccounts'].astype(str).str.lower() == 'true').astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — RELATIONSHIP NETWORK FEATURES
# ENRICHMENT ONLY — excluded from MODEL_FEATURES (see header for reason)
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding relationship features (enrichment only — NOT in model)...")

client_ids = set(party[party['ClientIndicator'] == 'Y']['PartyId'])
pr['related_is_client'] = pr['RelatedPartyId'].isin(client_ids).astype(int)

rel_feats = (pr[~pr['PartyId'].isin(client_ids)]
             .groupby('PartyId').agg(
                 n_client_relationships = ('related_is_client', 'sum'),
                 n_total_relationships  = ('RelatedPartyId',    'nunique'),
             ).reset_index())
rel_feats['pct_relations_are_clients'] = (
    rel_feats['n_client_relationships'] /
    rel_feats['n_total_relationships'].clip(lower=1)
)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — PARTY DEMOGRAPHICS
# Only PartyAge is reliably populated for non-clients.
# Marital status dummies are kept in the master table for inspection
# but are NOT added to MODEL_FEATURES (see header for reason).
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding demographic features...")

party['PartyAge']    = pd.to_numeric(party['PartyAge'], errors='coerce')
party['age_missing'] = party['PartyAge'].isna().astype(int)
party['PartyAge']    = party['PartyAge'].fillna(party['PartyAge'].median())

party['age_under_40'] = (party['PartyAge'] < 40).astype(int)
party['age_40_55']    = ((party['PartyAge'] >= 40) & (party['PartyAge'] < 55)).astype(int)
party['age_55_70']    = ((party['PartyAge'] >= 55) & (party['PartyAge'] < 70)).astype(int)
party['age_over_70']  = (party['PartyAge'] >= 70).astype(int)

# marital_* dummies — kept in master for inspection but NOT in MODEL_FEATURES
marital_dummies = pd.get_dummies(
    party['PartyMaritalStatus'], prefix='marital', dummy_na=False
).astype(int)

demo_cols = ['PartyId','ClientIndicator','PartyAge','age_missing',
             'age_under_40','age_40_55','age_55_70','age_over_70']
demo = pd.concat([party[demo_cols].reset_index(drop=True),
                  marital_dummies.reset_index(drop=True)], axis=1)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — ASSEMBLE MASTER TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\nAssembling master feature table...")

master = demo.copy()
master['PartyId'] = master['PartyId'].astype(str)

joins = [
    nc_roles,
    hh_feats,
    rel_feats,
    cm_feats,
    wmo[['PartyId','registered_in_wmo','has_aggregated_accounts']],
]
for df in joins:
    df['PartyId'] = df['PartyId'].astype(str)
    master = master.merge(df, on='PartyId', how='left')

num_cols = master.select_dtypes(include=[np.number]).columns
master[num_cols] = master[num_cols].fillna(0)
master['is_client'] = (master['ClientIndicator'] == 'Y').astype(int)

print(f"\nMaster table: {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"  Clients:     {master['is_client'].sum():,}")
print(f"  Non-clients: {(master['is_client']==0).sum():,}")

# Leakage check on ALL features (campaign/WMO/relationship will still be high —
# they are in master for enrichment but NOT in MODEL_FEATURES)
all_feat_check = [c for c in master.columns
                  if c not in ['PartyId','ClientIndicator','is_client']]
check = master.groupby('is_client')[all_feat_check].mean()
ratio = (check.loc[1] / (check.loc[0] + 1e-9)).sort_values(ascending=False)
print("\n=== Full leakage check (top 20) — enrichment features will still show high ratios ===")
print(ratio.head(20).round(1).to_string())

master.to_parquet(os.path.join(OUTPUT_PATH, 'master_modeling_table.parquet'), index=False)
print("\nMaster table saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — DEFINE MODEL FEATURES (LEAKAGE-SAFE)
#
# EXCLUDED from model (kept in master for output enrichment):
#   marital_*                    — only recorded for existing clients
#   n_campaigns_targeted         — CampaignMember = clients only
#   ever_responded               — CampaignMember = clients only
#   ever_opted_out_email         — CampaignMember = clients only
#   ever_do_not_call             — CampaignMember = clients only
#   registered_in_wmo            — only registered after becoming a client
#   has_aggregated_accounts      — only after becoming a client
#   n_client_relationships       — PartyRelationship only populated for clients
#   n_total_relationships        — PartyRelationship only populated for clients
#   pct_relations_are_clients    — derived from above
# ══════════════════════════════════════════════════════════════════════════════
MODEL_FEATURES = [
    # Demographics (PartyAge is populated for both clients and non-clients)
    'PartyAge', 'age_missing',
    'age_under_40', 'age_40_55', 'age_55_70', 'age_over_70',

    # Role counts on RBC accounts
    'n_accounts_named_on',
    'n_as_primary_bene', 'n_as_contingent_bene', 'n_as_tod_bene',
    'n_as_trusted_contact', 'n_as_minor', 'n_as_any_bene',
    'multi_account_named',

    # Portfolio value of accounts they are named on
    'associated_portfolio_total', 'associated_portfolio_max',
    'associated_portfolio_mean', 'associated_peak_portfolio',
    'log_portfolio', 'log_max_portfolio',

    # Account type signals
    'n_advisory_accts', 'n_ira_accts', 'n_trust_accts',
    'n_joint_accts', 'n_529_accts',
    'has_advisory_associated', 'has_ira_associated', 'has_trust_associated',
    'associated_acct_age_max', 'associated_acct_age_mean',

    # Inheritance proximity — key signal for beneficiaries
    'any_owner_70plus', 'any_owner_80plus', 'max_owner_age',

    # Money movement on associated accounts
    'associated_rollover_total', 'associated_distribution_total',
    'associated_has_rollover', 'associated_has_distribution',
    'associated_has_rmd',

    # Risk indicators
    'associated_margin_balance',
    'associated_any_federal_call', 'associated_any_house_call',

    # Household (genuine signal: non-clients can be family of a client)
    # Remove these two if AUC is still > 0.90 after other fixes
    'in_client_household', 'n_households',
]

MODEL_FEATURES = [f for f in MODEL_FEATURES if f in master.columns]
print(f"\nFinal model features: {len(MODEL_FEATURES)}")

# Leakage check on MODEL_FEATURES only — should all be < 50
model_check = master.groupby('is_client')[MODEL_FEATURES].mean()
model_ratio = (model_check.loc[1] / (model_check.loc[0] + 1e-9)).sort_values(ascending=False)
print("\n=== Leakage check on MODEL FEATURES only (target: all ratios < 50) ===")
print(model_ratio.head(15).round(1).to_string())

# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — TRAIN / VALIDATION / TEST SPLIT  (60 / 20 / 20)
# ══════════════════════════════════════════════════════════════════════════════
X         = master[MODEL_FEATURES].copy().fillna(0)
y         = master['is_client'].copy()
party_ids = master['PartyId'].copy()

X_tv, X_test, y_tv, y_test, pid_tv, pid_test = train_test_split(
    X, y, party_ids, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val, pid_train, pid_val = train_test_split(
    X_tv, y_tv, pid_tv, test_size=0.25, random_state=42, stratify=y_tv)

print(f"\nSplit summary:")
print(f"  Train:      {len(X_train):,}  ({y_train.mean():.2%} positive)")
print(f"  Validation: {len(X_val):,}   ({y_val.mean():.2%} positive)")
print(f"  Test:       {len(X_test):,}   ({y_test.mean():.2%} positive)")

nc_mask    = master['is_client'] == 0
X_score    = X[nc_mask].copy()
meta_score = master[nc_mask][[
    'PartyId', 'PartyAge',
    'n_accounts_named_on', 'associated_portfolio_total',
    'any_owner_70plus', 'any_owner_80plus',
    'n_advisory_accts', 'in_client_household',
]].copy()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — TRAIN XGBOOST
# ══════════════════════════════════════════════════════════════════════════════
print("\nTraining XGBoost...")

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"  scale_pos_weight = {spw:.2f}  ({neg:,} negatives / {pos:,} positives)")

model = XGBClassifier(
    n_estimators          = 500,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,
    scale_pos_weight      = spw,
    eval_metric           = 'auc',
    use_label_encoder     = False,
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 20,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"  Best iteration: {model.best_iteration}")

print("\nRunning 5-fold CV on train set...")
cv_model = XGBClassifier(
    n_estimators      = model.best_iteration or 200,
    max_depth=5, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=10,
    scale_pos_weight=spw, eval_metric='auc',
    use_label_encoder=False, random_state=42, n_jobs=-1,
)
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_apr = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
print(f"  CV ROC-AUC:  {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")
print(f"  CV Avg Prec: {cv_apr.mean():.4f} +/- {cv_apr.std():.4f}")

y_prob_train = model.predict_proba(X_train)[:, 1]
y_prob_val   = model.predict_proba(X_val)[:, 1]
y_prob_test  = model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_prob_train)
val_auc   = roc_auc_score(y_val,   y_prob_val)
test_auc  = roc_auc_score(y_test,  y_prob_test)
train_apr = average_precision_score(y_train, y_prob_train)
val_apr   = average_precision_score(y_val,   y_prob_val)
test_apr  = average_precision_score(y_test,  y_prob_test)

print(f"\n{'Split':<14} {'ROC-AUC':>10} {'Avg Precision':>15}")
print("─" * 42)
for name, auc, apr in [
    ('Train',      train_auc,     train_apr),
    ('CV mean',    cv_auc.mean(), cv_apr.mean()),
    ('Validation', val_auc,       val_apr),
    ('Test',       test_auc,      test_apr),
]:
    print(f"{name:<14} {auc:>10.4f} {apr:>15.4f}")

if test_auc > 0.90:
    print(f"\n[!] AUC = {test_auc:.4f} is still high (> 0.90).")
    print("    Check in_client_household in the model feature ratio table above.")
    print("    If its ratio is very high, remove 'in_client_household' and")
    print("    'n_households' from MODEL_FEATURES and re-run.")
else:
    print(f"\n[OK] AUC = {test_auc:.4f} — in realistic range, leakage removed.")

fpr_v, tpr_v, thr_v = roc_curve(y_val, y_prob_val)
best_thresh = float(thr_v[np.argmax(tpr_v - fpr_v)])
print(f"\nOptimal threshold (Youden's J on val): {best_thresh:.3f}")
print(classification_report(y_test, (y_prob_test >= best_thresh).astype(int)))

importance_df = pd.DataFrame({
    'feature':    MODEL_FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features:")
print(importance_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — SCORE NON-CLIENTS + ROLE ENRICHMENT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nScoring {len(X_score):,} non-client parties...")
conversion_prob = model.predict_proba(X_score)[:, 1]

ROLE_PRIORITY = [
    'Power of Attorney','Trustee','Guardian','Conservator',
    'Joint Owner','Authorized Signer','Trading Authority','Trusted Contact',
    'Primary Beneficiary','Contingent Beneficiary',
    'Transfer on Death (TOD) Bene Primary',
    'Transfer on Death (TOD) Bene Contingent',
    'Designated Beneficiary','Minor',
]

role_primary = (role.groupby('PartyId')
    .apply(lambda g: next(
        (r for r in ROLE_PRIORITY if r in g['FinancialAccountRole'].values),
        g['FinancialAccountRole'].iloc[0] if len(g) else 'No role'
    ))
    .reset_index().rename(columns={0: 'primary_role'}))

role_all = (role.groupby('PartyId')['FinancialAccountRole']
    .apply(lambda x: ', '.join(sorted(set(x))))
    .reset_index().rename(columns={'FinancialAccountRole': 'roles_held'}))

results = meta_score.copy()
results['conversion_probability'] = conversion_prob
results['priority_tier'] = pd.cut(
    results['conversion_probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)
results = (results
    .merge(role_primary, on='PartyId', how='left')
    .merge(role_all,     on='PartyId', how='left')
    .sort_values('conversion_probability', ascending=False)
    .reset_index(drop=True))
results['rank'] = results.index + 1

print(f"\nPriority tier distribution:")
print(results['priority_tier'].value_counts().sort_index().to_string())
print(f"\nTop 10 targets:")
print(results[['rank','PartyId','conversion_probability','priority_tier',
               'primary_role','associated_portfolio_total',
               'any_owner_70plus']].head(10).to_string(index=False))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 13 — K-MEANS CLUSTERING ON HIGH-PROBABILITY NON-CLIENTS
# ══════════════════════════════════════════════════════════════════════════════
HIGH_THRESH = 0.50
high_mask   = conversion_prob >= HIGH_THRESH
n_high      = high_mask.sum()
print(f"\nHigh-probability pool (>={HIGH_THRESH}): {n_high:,} parties")

do_clustering = n_high >= 30

if do_clustering:
    CLUSTER_FEATS = [f for f in [
        'PartyAge', 'age_under_40', 'age_40_55', 'age_55_70', 'age_over_70',
        'n_accounts_named_on',
        'n_as_primary_bene', 'n_as_contingent_bene', 'n_as_tod_bene',
        'n_as_trusted_contact', 'n_as_minor',
        'log_portfolio', 'log_max_portfolio',
        'n_advisory_accts', 'n_ira_accts', 'n_trust_accts',
        'any_owner_70plus', 'any_owner_80plus',
        'associated_has_rollover', 'associated_has_distribution',
        'in_client_household',
    ] if f in master.columns]

    X_high   = X_score[high_mask][CLUSTER_FEATS].fillna(0)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_high)

    max_k   = min(7, max(3, n_high // 100))
    K_range = range(2, max_k + 1)
    inertias = [KMeans(n_clusters=k, random_state=42, n_init=10)
                .fit(X_scaled).inertia_ for k in K_range]

    if len(inertias) >= 3:
        best_k = int(list(K_range)[np.argmax(np.diff(np.diff(inertias))) + 2])
        best_k = max(3, min(best_k, max_k))
    else:
        best_k = max_k

    print(f"Optimal k = {best_k}")
    km_final       = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    cluster_labels = km_final.fit_predict(X_scaled)

    X_high_prof = X_high.copy()
    X_high_prof['cluster']               = cluster_labels
    X_high_prof['conversion_probability'] = conversion_prob[high_mask]

    cluster_summary = X_high_prof.groupby('cluster').mean().round(3)
    cluster_summary['n_parties'] = X_high_prof.groupby('cluster').size()

    pop_mean = X_high_prof[CLUSTER_FEATS].mean()
    pop_std  = X_high_prof[CLUSTER_FEATS].std().replace(0, 1)

    LABEL_MAP = {
        'n_as_primary_bene':    'Primary Beneficiaries — Inheritance Ready',
        'n_as_tod_bene':        'TOD Beneficiaries — Estate Planning Stage',
        'n_as_trusted_contact': 'Trusted Contacts — Family Network',
        'n_as_minor':           'Minors — Long-term Growth Opportunity',
        'n_advisory_accts':     'Advisory-Adjacent — Upgrade Ready',
        'n_trust_accts':        'Trust-Associated Prospects',
        'any_owner_80plus':     'Near-Inheritance (Owner 80+)',
        'any_owner_70plus':     'Pre-Inheritance (Owner 70+)',
        'in_client_household':  'Client Household Members',
        'associated_has_rollover': 'Rollover / Transition Opportunity',
        'log_portfolio':        'High-Value Account Associates',
    }

    def name_cluster(row):
        feat_cols = [c for c in LABEL_MAP if c in row.index]
        if not feat_cols:
            return f'Segment {int(row.name)+1}'
        z = (row[feat_cols] - pop_mean[feat_cols]) / pop_std[feat_cols]
        best = z.idxmax()
        if z[best] < 0.25:
            return ('Older Wealth-Adjacent (60+)'
                    if row.get('age_55_70', 0) + row.get('age_over_70', 0) > 0.5
                    else 'General Prospect Pool')
        return LABEL_MAP[best]

    cluster_summary['segment_name'] = cluster_summary.apply(name_cluster, axis=1)
    name_map = cluster_summary['segment_name'].to_dict()

    print(f"\n{'Cluster':<5} {'Segment Name':<45} {'N':>7} {'Avg Prob':>10}")
    print("─" * 72)
    for i, row in cluster_summary.iterrows():
        print(f"{i:<5} {row['segment_name']:<45} "
              f"{int(row['n_parties']):>7,} {row['conversion_probability']:>10.3f}")

    pca     = PCA(n_components=2, random_state=42)
    X_pca   = pca.fit_transform(X_scaled)
    pca_var = pca.explained_variance_ratio_

    results_high = results[results['priority_tier'].isin(
        ['High','Very High'])].reset_index(drop=True).copy()
    results_high['cluster']      = cluster_labels
    results_high['segment_name'] = results_high['cluster'].map(name_map)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 14 — CHARTS
# ══════════════════════════════════════════════════════════════════════════════
print("\nGenerating charts...")

COLORS  = {'train':'#C04848','cv':'#1D9E75','val':'#E8A020','test':'#1B2B5E'}
PALETTE = ['#1B2B5E','#1D9E75','#E8A020','#C04848','#7B3F9E','#0E7B8C','#D4531E']

fig1, axes = plt.subplots(2, 3, figsize=(20, 12))
fig1.suptitle('RBC Beneficiary Conversion Model — Performance (Leakage-Fixed)',
              fontsize=15, fontweight='bold')
axes = axes.flatten()

ax = axes[0]
for lbl, yt, yp, c in [
    ('Train', y_train, y_prob_train, COLORS['train']),
    ('Val',   y_val,   y_prob_val,   COLORS['val']),
    ('Test',  y_test,  y_prob_test,  COLORS['test']),
]:
    fpr, tpr, _ = roc_curve(yt, yp)
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{lbl} ({roc_auc_score(yt,yp):.3f})')
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

ax = axes[1]
for lbl, yt, yp, c in [
    ('Train', y_train, y_prob_train, COLORS['train']),
    ('Val',   y_val,   y_prob_val,   COLORS['val']),
    ('Test',  y_test,  y_prob_test,  COLORS['test']),
]:
    pr_, rc_, _ = precision_recall_curve(yt, yp)
    ax.plot(rc_, pr_, color=c, lw=2, label=f'{lbl} ({average_precision_score(yt,yp):.3f})')
ax.axhline(y.mean(), color='gray', ls='--', lw=0.8, label='Baseline')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

ax = axes[2]
split_lbls = ['Train','CV','Val','Test']
auc_vals   = [train_auc, cv_auc.mean(), val_auc, test_auc]
bar_cols   = [COLORS['train'], COLORS['cv'], COLORS['val'], COLORS['test']]
bars = ax.bar(split_lbls, auc_vals, color=bar_cols, edgecolor='white', width=0.55)
for b, v in zip(bars, auc_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.errorbar(1, cv_auc.mean(), yerr=cv_auc.std()*2, fmt='none', color='black', capsize=5)
ax.set_ylim(0.4, 1.05); ax.set_ylabel('ROC-AUC')
ax.set_title('AUC by Split (CV +/-2SD)', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

ax = axes[3]
for lbl, yp, c in [
    ('Train', y_prob_train, COLORS['train']),
    ('Val',   y_prob_val,   COLORS['val']),
    ('Test',  y_prob_test,  COLORS['test']),
]:
    ax.hist(yp, bins=40, alpha=0.5, color=c, label=lbl, density=True)
ax.axvline(best_thresh, color='black', ls='--', lw=1.2, label=f'Threshold ({best_thresh:.2f})')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Density')
ax.set_title('Score Distributions', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

ax = axes[4]
sidx    = np.argsort(-y_prob_test)
sy      = np.array(y_test)[sidx]
n_pos   = sy.sum()
cum_pos = np.cumsum(sy)
pct_c   = np.arange(1, len(sy)+1) / len(sy) * 100
pct_cap = cum_pos / n_pos * 100
ax.plot(pct_c, pct_cap, color=COLORS['test'], lw=2, label='Model')
ax.plot([0,100],[0,100],'k--',lw=0.8,alpha=0.4,label='Random')
ax.fill_between(pct_c, pct_c, pct_cap, alpha=0.08, color=COLORS['test'])
i20 = np.searchsorted(pct_c, 20)
ax.annotate(f"Top 20% -> {pct_cap[i20]:.0f}% captured",
            xy=(20, pct_cap[i20]), xytext=(35, pct_cap[i20]-12),
            arrowprops=dict(arrowstyle='->', lw=1), fontsize=9)
ax.set_xlabel('% parties contacted'); ax.set_ylabel('% converters captured')
ax.set_title('Cumulative Gain (Test)', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

ax = axes[5]
top20 = importance_df.head(20)
fi_c  = [COLORS['test'] if i<5 else COLORS['cv'] if i<10 else '#A0AEC0' for i in range(len(top20))]
ax.barh(top20['feature'][::-1], top20['importance'][::-1], color=fi_c[::-1], edgecolor='white')
ax.set_xlabel('Importance (gain)')
ax.set_title('Top 20 Features', fontweight='bold')
ax.tick_params(axis='y', labelsize=8)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'model_performance_charts.png'), dpi=150, bbox_inches='tight')
plt.close()
print("  Performance charts saved.")

if do_clustering:
    fig2, axes2 = plt.subplots(2, 3, figsize=(20, 12))
    fig2.suptitle('RBC Prospect Segmentation — Role-Based Clusters', fontsize=15, fontweight='bold')
    axes2 = axes2.flatten()

    ax = axes2[0]
    ax.plot(list(K_range), inertias, 'o-', color=COLORS['test'], lw=2, markersize=6)
    ax.axvline(best_k, color=COLORS['cv'], ls='--', lw=1.5, label=f'k={best_k}')
    ax.set_xlabel('k'); ax.set_ylabel('Inertia')
    ax.set_title('Elbow Method', fontweight='bold')
    ax.legend(); ax.spines[['top','right']].set_visible(False)

    ax = axes2[1]
    for i in range(best_k):
        m = cluster_labels == i
        ax.scatter(X_pca[m,0], X_pca[m,1], c=PALETTE[i%len(PALETTE)],
                   alpha=0.5, s=15, label=name_map.get(i,'')[:25])
    ax.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)')
    ax.set_title('Cluster Map (PCA)', fontweight='bold')
    ax.legend(fontsize=7); ax.spines[['top','right']].set_visible(False)

    ax = axes2[2]
    sc = ax.scatter(X_pca[:,0], X_pca[:,1], c=conversion_prob[high_mask],
                    cmap='RdYlGn', alpha=0.5, s=15, vmin=0.5, vmax=1.0)
    plt.colorbar(sc, ax=ax, label='Conv. probability')
    ax.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)')
    ax.set_title('PCA — Conv. Probability', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes2[3]
    seg_names = [name_map.get(i,'')[:22] for i in range(best_k)]
    seg_sizes = [int(cluster_summary.loc[i,'n_parties']) for i in range(best_k)]
    seg_probs = [cluster_summary.loc[i,'conversion_probability'] for i in range(best_k)]
    xpos = np.arange(best_k)
    bars_d = ax.bar(xpos, seg_sizes,
                    color=[PALETTE[i%len(PALETTE)] for i in range(best_k)],
                    edgecolor='white', width=0.6)
    ax2t = ax.twinx()
    ax2t.plot(xpos, seg_probs, 'o--', color='black', lw=1.5, markersize=7)
    ax2t.set_ylabel('Mean conv. prob', fontsize=9); ax2t.set_ylim(0.4, 1.0)
    for b, s in zip(bars_d, seg_sizes):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
                f'{s:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(xpos)
    ax.set_xticklabels(seg_names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('# parties')
    ax.set_title('Segment Size & Avg Probability', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes2[4]
    box_data = [X_high_prof[X_high_prof['cluster']==i]['conversion_probability'].values
                for i in range(best_k)]
    bp = ax.boxplot(box_data, patch_artist=True)
    for patch, c in zip(bp['boxes'], PALETTE[:best_k]):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    for med in bp['medians']:
        med.set_color('white'); med.set_linewidth(2)
    ax.set_xticklabels([n[:18] for n in seg_names], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Conversion probability')
    ax.set_title('Probability per Segment', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes2[5]
    if 'primary_role' in results.columns:
        tier_order = ['Very High','High','Medium','Low']
        tr = (results.groupby(['priority_tier','primary_role']).size().unstack(fill_value=0))
        tr_pct = tr.div(tr.sum(axis=1), axis=0) * 100
        tr_pct = tr_pct.reindex([t for t in tier_order if t in tr_pct.index])
        tr_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='white', lw=0.5)
        ax.set_ylabel('% of parties')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10)
        ax.set_title('Role Mix by Priority Tier', fontweight='bold')
        ax.legend(fontsize=7, bbox_to_anchor=(1.6, 1.0))
        ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'clustering_charts.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("  Clustering charts saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 15 — SAVE ALL OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════
print("\nSaving outputs...")

results.to_csv(os.path.join(OUTPUT_PATH, 'conversion_scores_all.csv'), index=False)

priority = results[results['priority_tier'].isin(['Very High','High'])].copy()
if do_clustering:
    priority = priority.merge(
        results_high[['PartyId','cluster','segment_name']], on='PartyId', how='left')
priority.to_csv(os.path.join(OUTPUT_PATH, 'conversion_scores_priority.csv'), index=False)

results.head(1000).to_csv(os.path.join(OUTPUT_PATH, 'conversion_top1000.csv'), index=False)

if do_clustering:
    cluster_summary.reset_index().to_csv(
        os.path.join(OUTPUT_PATH, 'cluster_profiles.csv'), index=False)
    joblib.dump(scaler,   os.path.join(OUTPUT_PATH, 'cluster_scaler.pkl'))
    joblib.dump(km_final, os.path.join(OUTPUT_PATH, 'cluster_kmeans.pkl'))

joblib.dump(model, os.path.join(OUTPUT_PATH, 'conversion_model_xgb.pkl'))
importance_df.to_csv(os.path.join(OUTPUT_PATH, 'feature_importance.csv'), index=False)

pd.DataFrame({
    'split':         ['Train','CV_mean','CV_std','Validation','Test'],
    'roc_auc':       [train_auc, cv_auc.mean(), cv_auc.std(), val_auc, test_auc],
    'avg_precision': [train_apr, cv_apr.mean(), cv_apr.std(), val_apr, test_apr],
}).to_csv(os.path.join(OUTPUT_PATH, 'model_metrics.csv'), index=False)

cluster_str = ""
if do_clustering:
    cluster_str = f"\n  Clusters (k={best_k}):\n" + "\n".join(
        f"    {i}: {row['segment_name']} -- {int(row['n_parties']):,} parties"
        for i, row in cluster_summary.iterrows()
    )

print(f"""
{'='*65}
DONE
{'='*65}
  Test AUC:      {test_auc:.4f}
  Test Avg Prec: {test_apr:.4f}
  CV AUC:        {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}
  Threshold:     {best_thresh:.3f}

  Non-clients scored: {len(results):,}
    Very High (>0.70): {(results['priority_tier']=='Very High').sum():,}
    High (0.50-0.70):  {(results['priority_tier']=='High').sum():,}
    Medium (0.30-0.50):{(results['priority_tier']=='Medium').sum():,}
    Low (<0.30):       {(results['priority_tier']=='Low').sum():,}
{cluster_str}

  All outputs saved to:
    {OUTPUT_PATH}
{'='*65}
""")


Files found in Input folder: 26
History files found: 6
  BroadridgeAccountAdditionalHistory20250101to20250301.csv
  BroadridgeAccountAdditionalHistory20250301to20250501.csv
  BroadridgeAccountAdditionalHistory20250501to20250701.csv
  BroadridgeAccountAdditionalHistory20250701to20250901.csv
  BroadridgeAccountAdditionalHistory20250901to20251101.csv
  BroadridgeAccountAdditionalHistory20260101to20260225.csv

Loading source files...
Loading history files (may take ~2 mins on Drive)...
  Reading BroadridgeAccountAdditionalHistory20250101to20250301.csv...
  Reading BroadridgeAccountAdditionalHistory20250301to20250501.csv...
  Reading BroadridgeAccountAdditionalHistory20250501to20250701.csv...
  Reading BroadridgeAccountAdditionalHistory20250701to20250901.csv...
  Reading BroadridgeAccountAdditionalHistory20250901to20251101.csv...
  Reading BroadridgeAccountAdditionalHistory20260101to20260225.csv...
  Total history rows: 10,307,094

Parties:      155,154
Accounts:     83,652
Roles:        27

In [ ]:
import pandas as pd
import numpy as np

# Load the master table you saved
master = pd.read_parquet('/content/drive/MyDrive/RBC Capstone /Model Output/master_modeling_table.parquet')

feat_cols = [c for c in master.columns
             if c not in ['PartyId','ClientIndicator','is_client']]

# Check zero rates for non-clients
nc   = master[master['is_client'] == 0]
cl   = master[master['is_client'] == 1]

nc_zero_pct  = (nc[feat_cols] == 0).mean() * 100
cl_mean      = cl[feat_cols].mean()
nc_mean      = nc[feat_cols].mean()
ratio        = (cl_mean / (nc_mean + 1e-9))

diag = pd.DataFrame({
    'nc_zero_pct':  nc_zero_pct.round(1),
    'nc_mean':      nc_mean.round(4),
    'client_mean':  cl_mean.round(4),
    'ratio':        ratio.round(1),
}).sort_values('ratio', ascending=False)

print("=== Features where non-clients are 100% zero ===")
print(diag[diag['nc_zero_pct'] == 100].to_string())
print()
print("=== Features with ratio > 20 ===")
print(diag[diag['ratio'] > 20].to_string())

=== Features where non-clients are 100% zero ===
                           nc_zero_pct  nc_mean  client_mean         ratio
n_campaigns_targeted             100.0   0.0000       1.4141  1.414136e+09
ever_responded                   100.0   0.0000       0.1558  1.557609e+08
has_aggregated_accounts          100.0   0.0000       0.0211  2.110651e+07
ever_opted_out_email             100.0   0.0000       0.0048  4.779630e+06
marital_Domestic Partner         100.0   0.0005       0.0019  4.100000e+00
associated_any_house_call        100.0   0.0000       0.0000  0.000000e+00
n_client_relationships           100.0   0.0004       0.0000  0.000000e+00
pct_relations_are_clients        100.0   0.0002       0.0000  0.000000e+00
n_total_relationships            100.0   0.0007       0.0000  0.000000e+00
ever_do_not_call                 100.0   0.0000       0.0000  0.000000e+00

=== Features with ratio > 20 ===
                         nc_zero_pct  nc_mean  client_mean         ratio
n_campaigns_targete

In [ ]:
# import os

# # Walk your entire MyDrive and find any parquet files
# for root, dirs, files in os.walk('/content/drive/MyDrive'):
#     for f in files:
#         if f.endswith('.parquet'):
#             print(os.path.join(root, f))

In [ ]:
# # import pandas as pd
# # import numpy as np
# # import os
# # import joblib
# # import warnings
# # warnings.filterwarnings('ignore')

# # from sklearn.model_selection import StratifiedKFold, cross_val_score
# # from sklearn.metrics import (roc_auc_score, average_precision_score,
# #                               classification_report, confusion_matrix,
# #                               roc_curve, precision_recall_curve)
# # from sklearn.calibration import CalibratedClassifierCV
# # from xgboost import XGBClassifier
# # import matplotlib
# # matplotlib.use('Agg')
# # import matplotlib.pyplot as plt

# # # ── Paths ─────────────────────────────────────────────────────
# # INPUT_PATH  = r"C:\Users\91702\Documents\Natasha\MSBA\Capstone\Capstone Data\Feature Engineering Output"
# # OUTPUT_PATH = r"C:\Users\91702\Documents\Natasha\MSBA\Capstone\Capstone Data\Feature Engineering Output"

# # # ─────────────────────────────────────────────────────────────
# # # STEP 1 — LOAD MASTER TABLE
# # # ─────────────────────────────────────────────────────────────
# # print("Loading master modeling table...")
# # master = pd.read_parquet(os.path.join(INPUT_PATH, "master_modeling_table.parquet"))
# # print(f"Loaded: {master.shape[0]:,} rows, {master.shape[1]} columns")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 2 — DEFINE FEATURES
# # # ─────────────────────────────────────────────────────────────
# # # Base feature columns — all engineered signals
# # base_features = [
# #     # Demographics
# #     'PartyAge', 'age_missing',

# #     # Wealth (ordinal encoded)
# #     'networth_encoded', 'income_encoded',

# #     # Account activity
# #     'total_accounts', 'open_accounts', 'has_advisory_account',
# #     'tenure_years', 'account_type_diversity', 'has_held_away',

# #     # External assets (Yodlee)
# #     'has_external_account', 'n_external_accounts',
# #     'total_external_balance', 'has_external_401k', 'has_external_ira',

# #     # Money movement
# #     'total_cytd_inflow', 'total_cytd_outflow',
# #     'has_rollover', 'has_large_withdrawal',

# #     # Email engagement
# #     'emails_sent', 'emails_opened', 'open_rate',
# #     'click_rate', 'is_engager', 'is_clicker',

# #     # Roles & relationships
# #     'has_any_beneficiary', 'has_primary_beneficiary',
# #     'has_contingent_beneficiary', 'has_tod_beneficiary',
# #     'has_designated_beneficiary', 'total_beneficiary_pct',
# #     'is_joint_owner', 'is_trustee', 'has_trusted_contact',
# #     'has_power_of_attorney', 'has_trading_authority',
# #     'has_authorized_signer', 'has_minor', 'has_guardian',
# #     'has_participant', 'is_conservator', 'n_distinct_roles',

# #     # Historical portfolio data
# #     'hist_total_portfolio_value', 'hist_avg_portfolio_value',
# #     'hist_peak_portfolio_value', 'hist_portfolio_growth_pct',
# #     'hist_portfolio_growth_6m', 'hist_portfolio_volatility',
# #     'hist_equity_pct_mean', 'hist_equity_pct_min',
# #     'hist_cash_ratio', 'hist_ever_federal_call',
# #     'hist_ever_house_call', 'hist_is_margin_user',
# #     'hist_is_high_risk', 'hist_ytd_margin_interest',
# #     'hist_n_accounts_in_history',
# # ]

# # # Add marital status dummies (one-hot from feature engineering)
# # marital_cols = [c for c in master.columns if c.startswith('marital_')]
# # all_features = base_features + marital_cols

# # # Keep only features that actually exist in the master table
# # all_features = [f for f in all_features if f in master.columns]
# # print(f"\nFeatures used: {len(all_features)}")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 3 — PREPARE TRAIN / SCORE SETS
# # # ─────────────────────────────────────────────────────────────
# # # Training set = ALL parties (model learns what separates clients from non-clients)
# # # Scoring set  = ClientIndicator = N only (who to target for conversion)

# # X = master[all_features].copy()
# # y = master['is_client'].copy()

# # # Replace any remaining NaN with 0 (safety net)
# # X = X.fillna(0)

# # # Keep PartyId for the output scoring file
# # party_ids = master['PartyId'].copy()

# # # Separate the non-client parties for scoring
# # non_client_mask = master['is_client'] == 0
# # X_score    = X[non_client_mask].copy()
# # ids_score  = party_ids[non_client_mask].copy()
# # meta_score = master[non_client_mask][['PartyId','PartyAge','TotalNetWorth',
# #                                        'AnnualIncome','ClientIndicator']].copy()

# # print(f"Training set:  {len(X):,} rows ({y.sum():,} positives)")
# # print(f"Scoring set:   {len(X_score):,} non-client parties to score")
# # print(f"Class balance: {y.mean():.1%} positive (clients)")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 4 — TRAIN XGBOOST MODEL
# # # ─────────────────────────────────────────────────────────────
# # # scale_pos_weight handles the class imbalance
# # neg = (y == 0).sum()
# # pos = (y == 1).sum()
# # spw = neg / pos
# # print(f"\nscale_pos_weight = {spw:.2f} (handles {neg:,} negatives vs {pos:,} positives)")

# # model = XGBClassifier(
# #     n_estimators      = 500,
# #     max_depth         = 5,
# #     learning_rate     = 0.05,
# #     subsample         = 0.8,
# #     colsample_bytree  = 0.8,
# #     min_child_weight  = 10,
# #     scale_pos_weight  = spw,
# #     eval_metric       = 'auc',
# #     use_label_encoder = False,
# #     random_state      = 42,
# #     n_jobs            = -1,
# # )

# # # ── Cross-validation to measure performance ───────────────────
# # print("\nRunning 5-fold stratified cross-validation...")
# # cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
# # apr_scores = cross_val_score(model, X, y, cv=cv, scoring='average_precision', n_jobs=-1)

# # print(f"  ROC-AUC:           {auc_scores.mean():.4f}  (±{auc_scores.std():.4f})")
# # print(f"  Avg Precision:     {apr_scores.mean():.4f}  (±{apr_scores.std():.4f})")

# # # ── Train final model on full dataset ─────────────────────────
# # print("\nTraining final model on full dataset...")
# # model.fit(X, y)
# # print("Training complete.")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 5 — EVALUATE ON TRAINING SET (sanity check)
# # # ─────────────────────────────────────────────────────────────
# # y_prob_train = model.predict_proba(X)[:, 1]
# # train_auc    = roc_auc_score(y, y_prob_train)
# # train_apr    = average_precision_score(y, y_prob_train)
# # print(f"\nTraining set performance:")
# # print(f"  ROC-AUC:       {train_auc:.4f}")
# # print(f"  Avg Precision: {train_apr:.4f}")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 6 — SCORE ALL NON-CLIENT PARTIES
# # # ─────────────────────────────────────────────────────────────
# # print(f"\nScoring {len(X_score):,} non-client parties...")
# # conversion_prob = model.predict_proba(X_score)[:, 1]

# # # Build output table
# # results = meta_score.copy()
# # results['conversion_probability'] = conversion_prob

# # # Assign priority tiers based on probability
# # results['priority_tier'] = pd.cut(
# #     results['conversion_probability'],
# #     bins   = [0, 0.3, 0.5, 0.7, 1.0],
# #     labels = ['Low', 'Medium', 'High', 'Very High']
# # )

# # # Sort by probability descending
# # results = results.sort_values('conversion_probability', ascending=False).reset_index(drop=True)
# # results['rank'] = results.index + 1

# # print(f"\nPriority tier distribution:")
# # print(results['priority_tier'].value_counts().sort_index())
# # print(f"\nTop 10 conversion targets:")
# # print(results[['rank','PartyId','conversion_probability','priority_tier',
# #                'PartyAge','TotalNetWorth']].head(10).to_string(index=False))

# # # ─────────────────────────────────────────────────────────────
# # # STEP 7 — FEATURE IMPORTANCE
# # # ─────────────────────────────────────────────────────────────
# # importance_df = pd.DataFrame({
# #     'feature':   all_features,
# #     'importance': model.feature_importances_
# # }).sort_values('importance', ascending=False).head(20)

# # print(f"\nTop 20 most important features:")
# # for _, row in importance_df.iterrows():
# #     bar = '█' * int(row['importance'] * 500)
# #     print(f"  {row['feature']:45s}  {row['importance']:.4f}  {bar}")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 8 — SAVE OUTPUTS
# # # ─────────────────────────────────────────────────────────────
# # os.makedirs(OUTPUT_PATH, exist_ok=True)

# # # Full scored list
# # scored_path = os.path.join(OUTPUT_PATH, "conversion_scores_all.csv")
# # results.to_csv(scored_path, index=False)
# # print(f"\nFull scored list saved: {scored_path}")

# # # High priority only (Very High + High)
# # priority_path = os.path.join(OUTPUT_PATH, "conversion_scores_priority.csv")
# # priority = results[results['priority_tier'].isin(['Very High', 'High'])]
# # priority.to_csv(priority_path, index=False)
# # print(f"Priority targets saved: {priority_path}  ({len(priority):,} parties)")

# # # Top 1000 for immediate advisor action
# # top1000_path = os.path.join(OUTPUT_PATH, "conversion_top1000.csv")
# # results.head(1000).to_csv(top1000_path, index=False)
# # print(f"Top 1000 saved:         {top1000_path}")

# # # Model artifact
# # model_path = os.path.join(OUTPUT_PATH, "conversion_model_xgb.pkl")
# # joblib.dump(model, model_path)
# # print(f"Model saved:            {model_path}")

# # # Feature importance
# # fi_path = os.path.join(OUTPUT_PATH, "feature_importance.csv")
# # importance_df.to_csv(fi_path, index=False)
# # print(f"Feature importance:     {fi_path}")

# # # ─────────────────────────────────────────────────────────────
# # # STEP 9 — CHARTS
# # # ─────────────────────────────────────────────────────────────
# # fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# # fig.suptitle('RBC Conversion Model — Performance Overview', fontsize=14, fontweight='bold')

# # # Chart 1: Probability distribution
# # axes[0].hist(conversion_prob, bins=50, color='#1B2B5E', alpha=0.8, edgecolor='white')
# # axes[0].axvline(0.5, color='#E8A020', linestyle='--', linewidth=1.5, label='0.5 threshold')
# # axes[0].axvline(0.3, color='#1D9E75', linestyle='--', linewidth=1.5, label='0.3 threshold')
# # axes[0].set_xlabel('Conversion Probability', fontsize=11)
# # axes[0].set_ylabel('Number of Parties', fontsize=11)
# # axes[0].set_title('Distribution of Conversion Probabilities\n(Non-client parties only)', fontsize=11)
# # axes[0].legend()
# # axes[0].spines['top'].set_visible(False)
# # axes[0].spines['right'].set_visible(False)

# # # Chart 2: Feature importance
# # colors = ['#1B2B5E' if i < 5 else '#4A7EC7' if i < 10 else '#A0AEC0'
# #           for i in range(len(importance_df))]
# # axes[1].barh(importance_df['feature'][::-1],
# #              importance_df['importance'][::-1],
# #              color=colors[::-1], edgecolor='white')
# # axes[1].set_xlabel('Feature Importance (XGBoost gain)', fontsize=11)
# # axes[1].set_title('Top 20 Features Driving\nConversion Prediction', fontsize=11)
# # axes[1].spines['top'].set_visible(False)
# # axes[1].spines['right'].set_visible(False)

# # # Chart 3: Priority tier breakdown
# # tier_counts = results['priority_tier'].value_counts()
# # tier_colors = {'Very High':'#C0392B','High':'#E8A020','Medium':'#4A7EC7','Low':'#A0AEC0'}
# # bars = axes[2].bar(tier_counts.index, tier_counts.values,
# #                    color=[tier_colors.get(t,'#A0AEC0') for t in tier_counts.index],
# #                    edgecolor='white', linewidth=1.5)
# # for bar, val in zip(bars, tier_counts.values):
# #     axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
# #                  f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
# # axes[2].set_ylabel('Number of Parties', fontsize=11)
# # axes[2].set_title('Conversion Priority Tier Distribution\n(Non-client parties)', fontsize=11)
# # axes[2].spines['top'].set_visible(False)
# # axes[2].spines['right'].set_visible(False)

# # plt.tight_layout()
# # chart_path = os.path.join(OUTPUT_PATH, "conversion_model_charts.png")
# # plt.show()
# # plt.savefig(chart_path, dpi=150, bbox_inches='tight')
# # plt.close()
# # print(f"Charts saved:           {chart_path}")

# # print(f"""
# # {'='*60}
# # SUMMARY
# # {'='*60}
# # CV ROC-AUC:           {auc_scores.mean():.4f}
# # CV Avg Precision:     {apr_scores.mean():.4f}

# # Non-client parties scored: {len(results):,}
# #   Very High priority:  {(results['priority_tier']=='Very High').sum():,}
# #   High priority:       {(results['priority_tier']=='High').sum():,}
# #   Medium priority:     {(results['priority_tier']=='Medium').sum():,}
# #   Low priority:        {(results['priority_tier']=='Low').sum():,}

# # Files saved to: {OUTPUT_PATH}
# #   conversion_scores_all.csv      — all non-client parties ranked
# #   conversion_scores_priority.csv — High + Very High only
# #   conversion_top1000.csv         — immediate advisor action list
# #   conversion_model_xgb.pkl       — model artifact
# #   feature_importance.csv         — feature importance table
# #   conversion_model_charts.png    — performance charts
# # {'='*60}
# # """)

# # ══════════════════════════════════════════════════════════════════════════════
# # RBC CONVERSION MODEL — COLAB-READY, LEAKAGE-FIXED VERSION
# # ══════════════════════════════════════════════════════════════════════════════
# #
# # COLAB SETUP INSTRUCTIONS (run these in a separate cell before this script):
# #
# #   # Cell 1 — install packages not pre-installed on Colab
# #   !pip install xgboost --quiet
# #
# #   # Cell 2 — mount your Google Drive (a browser popup will ask you to sign in)
# #   from google.colab import drive
# #   drive.mount('/content/drive')
# #
# #   # Cell 3 — upload your parquet file to Google Drive, then set this path.
# #   # Example: if your file is in "My Drive > Capstone Data", the path is:
# #   # /content/drive/MyDrive/Capstone Data/master_modeling_table.parquet
# #
# # ══════════════════════════════════════════════════════════════════════════════

# import pandas as pd
# import numpy as np
# import os
# import joblib
# import warnings
# warnings.filterwarnings('ignore')

# from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
# from sklearn.metrics import (roc_auc_score, average_precision_score,
#                               classification_report, roc_curve,
#                               precision_recall_curve)
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# from sklearn.decomposition import PCA
# from xgboost import XGBClassifier
# import matplotlib
# matplotlib.use('Agg')
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec

# # ──────────────────────────────────────────────────────────────────────────────
# # PATHS — edit INPUT_PATH to match where you uploaded the parquet on Drive
# # ──────────────────────────────────────────────────────────────────────────────
# INPUT_PATH  = '/content'   # ← change if needed
# OUTPUT_PATH = '/content/drive/MyDrive/RBC Capstone/Model Output'
# os.makedirs(OUTPUT_PATH, exist_ok=True)

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 1 — LOAD DATA
# # ──────────────────────────────────────────────────────────────────────────────
# print("Loading master modeling table...")
# master = pd.read_parquet(os.path.join(INPUT_PATH, "master_modeling_table.parquet"))
# print(f"Loaded: {master.shape[0]:,} rows × {master.shape[1]} columns")
# print(f"Class balance: {master['is_client'].mean():.2%} positive (clients)")

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 2 — LEAKAGE-SAFE FEATURE DEFINITION
# #
# # WHY WE EXCLUDE ROLE COLUMNS FROM MODEL FEATURES:
# # Columns like has_primary_beneficiary, is_trustee, has_power_of_attorney
# # describe a party's formal legal relationship with an EXISTING RBC account.
# # A beneficiary IS connected to a client's account by definition — so the
# # model learns "connected to RBC account = likely to convert", which is
# # circular and not a genuine predictive signal.
# #
# # These role columns are kept for OUTPUT ENRICHMENT (telling advisors the
# # relationship context) and for CLUSTERING (segmenting by relationship type),
# # but they must NOT be model features.
# #
# # Similarly, hist_* columns should be reviewed: if they are only populated
# # for parties who hold accounts (non-zero only for existing account holders),
# # they may also leak. Check with:
# #   master.groupby('is_client')[hist_cols].mean()
# # If means are near-identical across is_client=0 and is_client=1, safe to keep.
# # If hist cols are zero for most non-clients, remove them too.
# # ──────────────────────────────────────────────────────────────────────────────

# # Safe features: behavioural signals, demographics, external/held-away data,
# # email engagement, money movement — things observable BEFORE conversion
# SAFE_FEATURES = [
#     # Demographics
#     'PartyAge', 'age_missing',

#     # Wealth indicators (externally sourced or self-reported — not account-derived)
#     'networth_encoded', 'income_encoded',

#     # Account structure (counts and diversity, not role flags)
#     'total_accounts', 'open_accounts',
#     'tenure_years', 'account_type_diversity',

#     # External / held-away assets (Yodlee — strong leakage-free signal)
#     'has_external_account', 'n_external_accounts',
#     'total_external_balance', 'has_external_401k', 'has_external_ira',
#     'has_held_away',

#     # Money movement (CYTD — behavioural, not account-status derived)
#     'total_cytd_inflow', 'total_cytd_outflow',
#     'has_rollover', 'has_large_withdrawal',

#     # Email engagement (pure behavioural signal)
#     'emails_sent', 'emails_opened', 'open_rate',
#     'click_rate', 'is_engager', 'is_clicker',

#     # Historical portfolio signals — KEEP only if populated for non-clients too
#     # If these are zero/null for all non-clients, comment them out
#     'hist_total_portfolio_value', 'hist_avg_portfolio_value',
#     'hist_peak_portfolio_value', 'hist_portfolio_growth_pct',
#     'hist_portfolio_growth_6m', 'hist_portfolio_volatility',
#     'hist_equity_pct_mean', 'hist_equity_pct_min',
#     'hist_cash_ratio', 'hist_ever_federal_call',
#     'hist_ever_house_call', 'hist_is_margin_user',
#     'hist_is_high_risk', 'hist_ytd_margin_interest',
#     'hist_n_accounts_in_history',
# ]

# # EXCLUDED FROM MODEL (kept for output enrichment + clustering only):
# # has_primary_beneficiary, has_contingent_beneficiary, has_tod_beneficiary,
# # has_designated_beneficiary, is_joint_owner, is_trustee, has_trusted_contact,
# # has_power_of_attorney, has_trading_authority, has_authorized_signer,
# # has_minor, has_guardian, has_participant, is_conservator, n_distinct_roles,
# # has_any_beneficiary, total_beneficiary_pct, has_advisory_account

# marital_cols = [c for c in master.columns if c.startswith('marital_')]
# all_features = [f for f in SAFE_FEATURES + marital_cols if f in master.columns]
# print(f"\nSafe features used in model: {len(all_features)}")

# # ── Quick leakage check on hist columns ───────────────────────────────────────
# hist_cols = [f for f in all_features if f.startswith('hist_')]
# if hist_cols:
#     print("\nLeakage check — mean of hist columns by is_client:")
#     print(master.groupby('is_client')[hist_cols[:5]].mean().round(3).to_string())
#     print("  → If non-client (0) means are all zero and client (1) means are non-zero,")
#     print("    comment out the hist_* block in SAFE_FEATURES above.\n")

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 3 — RELATIONSHIP ROLE ENRICHMENT (for output + clustering only)
# # ──────────────────────────────────────────────────────────────────────────────
# ROLE_MAP = {
#     'has_primary_beneficiary':    'Primary Beneficiary',
#     'has_contingent_beneficiary': 'Contingent Beneficiary',
#     'has_tod_beneficiary':        'TOD Beneficiary',
#     'has_designated_beneficiary': 'Designated Beneficiary',
#     'is_joint_owner':             'Joint Owner',
#     'is_trustee':                 'Trustee',
#     'has_trusted_contact':        'Trusted Contact',
#     'has_power_of_attorney':      'Power of Attorney',
#     'has_trading_authority':      'Trading Authority',
#     'has_authorized_signer':      'Authorized Signer',
#     'has_minor':                  'Minor',
#     'has_guardian':               'Guardian',
#     'has_participant':            'Participant',
#     'is_conservator':             'Conservator',
# }
# existing_role_cols = {c: l for c, l in ROLE_MAP.items() if c in master.columns}
# print(f"Role columns found: {list(existing_role_cols.keys())}")

# def build_roles_held(row):
#     roles = [l for c, l in existing_role_cols.items() if row.get(c, 0)]
#     return ', '.join(roles) if roles else 'No secondary role'

# def primary_role(row):
#     priority = [
#         ('has_power_of_attorney',      'Power of Attorney'),
#         ('is_trustee',                 'Trustee'),
#         ('has_guardian',               'Guardian'),
#         ('is_conservator',             'Conservator'),
#         ('is_joint_owner',             'Joint Owner'),
#         ('has_authorized_signer',      'Authorized Signer'),
#         ('has_trading_authority',      'Trading Authority'),
#         ('has_trusted_contact',        'Trusted Contact'),
#         ('has_primary_beneficiary',    'Primary Beneficiary'),
#         ('has_contingent_beneficiary', 'Contingent Beneficiary'),
#         ('has_tod_beneficiary',        'TOD Beneficiary'),
#         ('has_designated_beneficiary', 'Designated Beneficiary'),
#         ('has_participant',            'Participant'),
#         ('has_minor',                  'Minor'),
#     ]
#     for col, label in priority:
#         if col in master.columns and row.get(col, 0):
#             return label
#     return 'No role / Account holder only'

# print("Building role summaries...")
# master['roles_held']   = master.apply(build_roles_held, axis=1)
# master['primary_role'] = master.apply(primary_role,     axis=1)

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 4 — TRAIN / VALIDATION / TEST SPLIT (60 / 20 / 20)
# # ──────────────────────────────────────────────────────────────────────────────
# X         = master[all_features].copy().fillna(0)
# y         = master['is_client'].copy()
# party_ids = master['PartyId'].copy()

# X_trainval, X_test, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
#     X, y, party_ids, test_size=0.20, random_state=42, stratify=y
# )
# X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
#     X_trainval, y_trainval, idx_trainval,
#     test_size=0.25, random_state=42, stratify=y_trainval
# )

# print(f"\nSplit summary:")
# print(f"  Train:      {len(X_train):,} rows  ({y_train.mean():.2%} positive)")
# print(f"  Validation: {len(X_val):,}  rows  ({y_val.mean():.2%} positive)")
# print(f"  Test:       {len(X_test):,}  rows  ({y_test.mean():.2%} positive)")

# non_client_mask = master['is_client'] == 0
# X_score    = X[non_client_mask].copy()
# meta_score = master[non_client_mask][[
#     'PartyId', 'PartyAge', 'TotalNetWorth', 'AnnualIncome',
#     'ClientIndicator', 'roles_held', 'primary_role'
# ]].copy()
# print(f"\nScoring set (non-clients): {len(X_score):,} parties")

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 5 — TRAIN XGBOOST
# # ──────────────────────────────────────────────────────────────────────────────
# neg = (y_train == 0).sum()
# pos = (y_train == 1).sum()
# spw = neg / pos
# print(f"\nscale_pos_weight = {spw:.2f}  ({neg:,} negatives / {pos:,} positives)")

# model = XGBClassifier(
#     n_estimators          = 500,
#     max_depth             = 5,
#     learning_rate         = 0.05,
#     subsample             = 0.8,
#     colsample_bytree      = 0.8,
#     min_child_weight      = 10,
#     scale_pos_weight      = spw,
#     eval_metric           = 'auc',
#     use_label_encoder     = False,
#     random_state          = 42,
#     n_jobs                = -1,
#     early_stopping_rounds = 20,
# )
# print("\nTraining model (early stopping on validation AUC)...")
# model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
# print(f"Best iteration: {model.best_iteration}")

# # ── Cross-validation ──────────────────────────────────────────────────────────
# print("\nRunning 5-fold CV on train set...")
# cv_model = XGBClassifier(
#     n_estimators      = model.best_iteration or 300,
#     max_depth         = 5, learning_rate=0.05, subsample=0.8,
#     colsample_bytree  = 0.8, min_child_weight=10,
#     scale_pos_weight  = spw, eval_metric='auc',
#     use_label_encoder = False, random_state=42, n_jobs=-1,
# )
# cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv_auc = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring='roc_auc',           n_jobs=-1)
# cv_apr = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
# print(f"  CV ROC-AUC:       {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
# print(f"  CV Avg Precision: {cv_apr.mean():.4f} ± {cv_apr.std():.4f}")

# # ── Evaluate ──────────────────────────────────────────────────────────────────
# y_prob_train = model.predict_proba(X_train)[:, 1]
# y_prob_val   = model.predict_proba(X_val)[:, 1]
# y_prob_test  = model.predict_proba(X_test)[:, 1]

# train_auc = roc_auc_score(y_train, y_prob_train)
# val_auc   = roc_auc_score(y_val,   y_prob_val)
# test_auc  = roc_auc_score(y_test,  y_prob_test)
# train_apr = average_precision_score(y_train, y_prob_train)
# val_apr   = average_precision_score(y_val,   y_prob_val)
# test_apr  = average_precision_score(y_test,  y_prob_test)

# print(f"\n{'Split':<14} {'ROC-AUC':>10} {'Avg Precision':>15}")
# print(f"{'─'*42}")
# for name, auc, apr in [
#     ('Train',      train_auc,      train_apr),
#     ('CV mean',    cv_auc.mean(),  cv_apr.mean()),
#     ('Validation', val_auc,        val_apr),
#     ('Test',       test_auc,       test_apr),
# ]:
#     print(f"{name:<14} {auc:>10.4f} {apr:>15.4f}")

# # ── Sanity check: flag suspiciously high performance ─────────────────────────
# if train_auc > 0.97:
#     print("\n⚠️  WARNING: Train AUC > 0.97 suggests data leakage.")
#     print("   Review the leakage check output above and remove any hist_*")
#     print("   columns that are zero for all non-clients.")
# if test_auc > 0.95:
#     print("\n⚠️  WARNING: Test AUC > 0.95 is unusually high for a conversion model.")
#     print("   A realistic range is 0.70–0.85. Check feature distributions.")

# fpr_v, tpr_v, thresholds_v = roc_curve(y_val, y_prob_val)
# best_thresh = float(thresholds_v[np.argmax(tpr_v - fpr_v)])
# print(f"\nOptimal threshold (Youden's J on val): {best_thresh:.3f}")
# y_pred_test = (y_prob_test >= best_thresh).astype(int)
# print("\nTest set classification report:")
# print(classification_report(y_test, y_pred_test))

# # ── Feature importance ────────────────────────────────────────────────────────
# importance_df = pd.DataFrame({
#     'feature': all_features, 'importance': model.feature_importances_
# }).sort_values('importance', ascending=False).head(20)

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 6 — SCORE NON-CLIENTS
# # ──────────────────────────────────────────────────────────────────────────────
# print(f"\nScoring {len(X_score):,} non-client parties...")
# conversion_prob = model.predict_proba(X_score)[:, 1]

# results = meta_score.copy()
# results['conversion_probability'] = conversion_prob
# results['priority_tier'] = pd.cut(
#     results['conversion_probability'],
#     bins=[0, 0.3, 0.5, 0.7, 1.0],
#     labels=['Low', 'Medium', 'High', 'Very High']
# )
# results = results.sort_values('conversion_probability', ascending=False).reset_index(drop=True)
# results['rank'] = results.index + 1
# results = results[[
#     'rank', 'PartyId', 'conversion_probability', 'priority_tier',
#     'primary_role', 'roles_held',
#     'PartyAge', 'TotalNetWorth', 'AnnualIncome', 'ClientIndicator'
# ]]

# print(f"\nPriority tier distribution:")
# print(results['priority_tier'].value_counts().sort_index())
# print(f"\nTop 10 targets:")
# print(results[['rank','PartyId','conversion_probability','priority_tier','primary_role']]
#       .head(10).to_string(index=False))

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 7 — K-MEANS CLUSTERING ON HIGH-PROBABILITY NON-CLIENTS
# #
# # CLUSTER DESIGN RATIONALE:
# # We cluster only on relationship role flags + demographics.
# # Role flags are NOT in the model (leakage prevention) but ARE meaningful
# # for segmentation — they tell us the relationship context driving conversion.
# # This answers: "among parties likely to convert, who ARE they?"
# #
# # MINIMUM SIZE GUARD:
# # k-means on fewer than ~50 parties produces unreliable segments.
# # We check the pool size and fall back gracefully if needed.
# # ──────────────────────────────────────────────────────────────────────────────

# HIGH_PROB_THRESHOLD = 0.50
# high_prob_mask = conversion_prob >= HIGH_PROB_THRESHOLD
# n_high         = high_prob_mask.sum()
# print(f"\nHigh-probability pool (prob ≥ {HIGH_PROB_THRESHOLD}): {n_high:,} parties")

# if n_high < 20:
#     print("⚠️  Fewer than 20 high-probability parties — clustering skipped.")
#     print("   This is a strong sign of remaining leakage or a very small dataset.")
#     print("   Fix leakage first, then re-run.")
#     do_clustering = False
# else:
#     do_clustering = True

# if do_clustering:
#     # Role + demographic features for clustering
#     CLUSTER_FEATURES = [f for f in [
#         'PartyAge', 'networth_encoded', 'income_encoded', 'tenure_years',
#         'has_primary_beneficiary', 'has_contingent_beneficiary',
#         'has_tod_beneficiary', 'is_joint_owner', 'is_trustee',
#         'has_power_of_attorney', 'has_trading_authority',
#         'has_trusted_contact', 'has_minor', 'has_guardian',
#         'is_conservator', 'n_distinct_roles', 'has_any_beneficiary',
#         'total_external_balance', 'total_cytd_inflow',
#         'hist_total_portfolio_value',
#     ] if f in master.columns]

#     X_high    = X_score[high_prob_mask].copy()
#     # For clustering we also need the role columns which aren't in X_score
#     # (they were excluded from model features) — pull from master directly
#     role_flag_cols = [c for c in existing_role_cols.keys() if c in master.columns]
#     extra_cols     = ['n_distinct_roles','has_any_beneficiary','total_beneficiary_pct']
#     extra_cols     = [c for c in extra_cols if c in master.columns]
#     master_high    = master[non_client_mask][high_prob_mask].copy()

#     X_cluster = pd.DataFrame(index=X_high.index)
#     for f in CLUSTER_FEATURES:
#         if f in X_high.columns:
#             X_cluster[f] = X_high[f].values
#         elif f in master_high.columns:
#             X_cluster[f] = master_high[f].values
#         # skip if not available in either

#     X_cluster = X_cluster.fillna(0)
#     print(f"Clustering features: {list(X_cluster.columns)}")

#     # Scale
#     scaler   = StandardScaler()
#     X_scaled = scaler.fit_transform(X_cluster)

#     # Elbow method — max k is capped at n_high//5 to ensure meaningful clusters
#     max_k    = min(8, n_high // 5)
#     max_k    = max(max_k, 3)
#     K_range  = range(2, max_k + 1)
#     inertias = []
#     for k in K_range:
#         km = KMeans(n_clusters=k, random_state=42, n_init=10)
#         km.fit(X_scaled)
#         inertias.append(km.inertia_)

#     if len(inertias) >= 3:
#         diffs  = np.diff(inertias)
#         diffs2 = np.diff(diffs)
#         best_k = int(list(K_range)[np.argmax(diffs2) + 2])
#         best_k = max(3, min(best_k, max_k))
#     else:
#         best_k = max_k
#     print(f"Optimal k = {best_k}")

#     km_final       = KMeans(n_clusters=best_k, random_state=42, n_init=20)
#     cluster_labels = km_final.fit_predict(X_scaled)

#     # Attach to high-prob results
#     results_high_mask = results['priority_tier'].isin(['High','Very High'])
#     results_high      = results[results_high_mask].copy()
#     results_high      = results_high.reset_index(drop=True)
#     results_high['cluster'] = cluster_labels

#     # ── Profile each cluster ──────────────────────────────────────────────────
#     cluster_profile_df                         = X_cluster.copy()
#     cluster_profile_df['cluster']              = cluster_labels
#     cluster_profile_df['conversion_probability'] = conversion_prob[high_prob_mask]

#     profile_cols = [c for c in [
#         'conversion_probability', 'PartyAge', 'networth_encoded',
#         'income_encoded', 'tenure_years',
#         'has_primary_beneficiary', 'has_contingent_beneficiary',
#         'has_tod_beneficiary', 'is_joint_owner', 'is_trustee',
#         'has_power_of_attorney', 'has_trading_authority',
#         'has_trusted_contact', 'has_minor', 'has_guardian',
#         'is_conservator', 'n_distinct_roles', 'has_any_beneficiary',
#         'total_external_balance', 'total_cytd_inflow',
#         'hist_total_portfolio_value',
#     ] if c in cluster_profile_df.columns]

#     cluster_summary = (
#         cluster_profile_df.groupby('cluster')[profile_cols]
#         .mean().round(3)
#     )
#     cluster_summary['n_parties'] = cluster_profile_df.groupby('cluster').size()

#     # ── Name clusters by their single most elevated role signal ───────────────
#     # Strategy: for each cluster, find which role flag is furthest above the
#     # population mean (in standard deviation units). That's the defining trait.
#     # Then apply a business-meaningful label.

#     role_flag_profile_cols = [c for c in [
#         'has_primary_beneficiary', 'has_contingent_beneficiary',
#         'has_tod_beneficiary', 'is_joint_owner', 'is_trustee',
#         'has_power_of_attorney', 'has_trading_authority',
#         'has_trusted_contact', 'has_minor', 'has_guardian',
#         'is_conservator', 'n_distinct_roles',
#     ] if c in cluster_summary.columns]

#     pop_mean = cluster_profile_df[role_flag_profile_cols].mean()
#     pop_std  = cluster_profile_df[role_flag_profile_cols].std().replace(0, 1)

#     ROLE_LABEL_MAP = {
#         'has_power_of_attorney':      'Legal Authority Holders — POA',
#         'is_trustee':                 'Trust Relationship Holders',
#         'has_guardian':               'Guardians & Conservators',
#         'is_conservator':             'Guardians & Conservators',
#         'is_joint_owner':             'Joint Account Holders',
#         'has_trading_authority':      'Delegated Trading Authority',
#         'has_trusted_contact':        'Trusted Contact Designees',
#         'has_primary_beneficiary':    'Primary Beneficiaries',
#         'has_contingent_beneficiary': 'Contingent Beneficiaries',
#         'has_tod_beneficiary':        'TOD / Estate-Planning Prospects',
#         'has_minor':                  'Families with Minor Accounts',
#         'n_distinct_roles':           'Multi-Role Complex Relationships',
#     }

#     def name_cluster(cluster_row):
#         if not role_flag_profile_cols:
#             return f"Segment {int(cluster_row.name) + 1}"
#         # Z-score each role for this cluster relative to the population
#         z_scores = (cluster_row[role_flag_profile_cols] - pop_mean) / pop_std
#         best_col = z_scores.idxmax()
#         best_z   = z_scores[best_col]
#         if best_z < 0.3:
#             # No role is distinctively elevated — fall back to demographic
#             age = cluster_row.get('PartyAge', 0)
#             nw  = cluster_row.get('networth_encoded', 0)
#             if nw > cluster_summary['networth_encoded'].median() * 1.2 if 'networth_encoded' in cluster_summary.columns else False:
#                 return 'High-Net-Worth Prospects'
#             if age > 55:
#                 return 'Older Wealth Accumulators (55+)'
#             return 'Engaged Non-Role Holders'
#         return ROLE_LABEL_MAP.get(best_col, f"Segment {int(cluster_row.name) + 1}")

#     cluster_summary['segment_name'] = cluster_summary.apply(name_cluster, axis=1)
#     name_map = cluster_summary['segment_name'].to_dict()

#     print("\nCluster segment profiles:")
#     print(f"{'Cluster':<5} {'Segment Name':<40} {'N':>6} {'Avg Prob':>10}")
#     print("─" * 65)
#     for i, row in cluster_summary.iterrows():
#         print(f"{i:<5} {row['segment_name']:<40} {int(row['n_parties']):>6} "
#               f"{row['conversion_probability']:>10.3f}")

#     # Explain each segment in detail
#     print("\n" + "═"*65)
#     print("SEGMENT EXPLANATIONS")
#     print("═"*65)
#     segment_explanations = {
#         'Legal Authority Holders — POA': (
#             "These parties hold Power of Attorney over an RBC account holder. "
#             "They already make financial decisions on behalf of someone at RBC. "
#             "Advisor angle: they understand wealth management intimately and may "
#             "want to consolidate their own assets under RBC."
#         ),
#         'Trust Relationship Holders': (
#             "Trustees on RBC accounts — they manage trust assets and understand "
#             "fiduciary responsibility. High likelihood of having their own "
#             "investable assets. Angle: offer to manage both the trust and their "
#             "personal portfolio under one advisory relationship."
#         ),
#         'Guardians & Conservators': (
#             "Legal guardians or conservators of a minor or incapacitated person "
#             "with an RBC account. They're in a planning-intensive life stage. "
#             "Angle: financial planning for dependents, education savings (529), "
#             "and eventually managing inherited assets."
#         ),
#         'Joint Account Holders': (
#             "Share an account with an existing RBC client — typically a spouse "
#             "or family member. Already familiar with RBC but haven't opened their "
#             "own account. Angle: easy conversion — they're one step away. Pitch "
#             "independent accounts, beneficiary planning, separate advisory."
#         ),
#         'TOD / Estate-Planning Prospects': (
#             "Named on a Transfer-on-Death designation — they'll inherit assets "
#             "from an existing RBC client. Likely older, inheritance-aware. "
#             "Angle: proactive estate planning conversation, pre-inheritance "
#             "portfolio setup, and tax-efficient transfer strategies."
#         ),
#         'Primary Beneficiaries': (
#             "Named as primary beneficiary on an RBC account. They will receive "
#             "assets upon the account holder's death. Angle: prepare them now — "
#             "open an account so the transfer is seamless, discuss what they "
#             "plan to do with the inheritance."
#         ),
#         'Contingent Beneficiaries': (
#             "Second-in-line beneficiary — less immediate than primary, but still "
#             "connected. Angle: similar to primary beneficiaries but lower urgency. "
#             "Good for email nurture; escalate when the primary beneficiary changes "
#             "or the account holder ages."
#         ),
#         'Families with Minor Accounts': (
#             "A minor holds an account at RBC — implying a parent or guardian is "
#             "managing it. The parent is the conversion target. Angle: education "
#             "savings, custodial accounts, and growing the family's full financial "
#             "relationship with RBC as the child approaches adulthood."
#         ),
#         'Trusted Contact Designees': (
#             "Listed as a trusted contact — someone RBC can reach out to if there "
#             "are concerns about the account holder's wellbeing. They're trusted "
#             "by an existing client. Angle: gentle outreach, relationship-first. "
#             "They may be a family member or close friend who isn't yet a client."
#         ),
#         'Delegated Trading Authority': (
#             "Given authority to trade on someone else's account. Financially "
#             "sophisticated and already active within the RBC platform. "
#             "Angle: they understand investing — pitch advisory services or "
#             "their own brokerage account."
#         ),
#         'Multi-Role Complex Relationships': (
#             "Hold multiple different roles across RBC accounts — trustee on one, "
#             "beneficiary on another, joint owner on a third. Deeply embedded in "
#             "the RBC ecosystem. High conversion probability because they interact "
#             "with RBC constantly. Angle: consolidation pitch — bring everything "
#             "under one personal advisory relationship."
#         ),
#         'High-Net-Worth Prospects': (
#             "No dominant role signal but elevated net worth / income. Likely "
#             "a wealthy individual connected to RBC through a family member's "
#             "account. Angle: direct wealth management and advisory pitch."
#         ),
#         'Older Wealth Accumulators (55+)': (
#             "Older parties without a strong role signal. In or near retirement "
#             "planning stage. Angle: retirement income, RMD planning, and "
#             "estate transition conversations."
#         ),
#         'Engaged Non-Role Holders': (
#             "No dominant role, but high email engagement or inflow activity "
#             "drove their conversion score. They're paying attention. "
#             "Angle: digital journey — follow up email engagement with a "
#             "targeted offer rather than a cold advisor call."
#         ),
#     }

#     for i, row in cluster_summary.iterrows():
#         name = row['segment_name']
#         expl = segment_explanations.get(name, "Review cluster profile for interpretation.")
#         print(f"\nCluster {i}: {name}  (n={int(row['n_parties']):,}, avg prob={row['conversion_probability']:.3f})")
#         print(f"  {expl}")

#     results_high['segment_name'] = results_high['cluster'].map(name_map)

#     # PCA for visualisation
#     pca     = PCA(n_components=2, random_state=42)
#     X_pca   = pca.fit_transform(X_scaled)
#     pca_var = pca.explained_variance_ratio_

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 8 — CHARTS
# # ──────────────────────────────────────────────────────────────────────────────
# COLORS = {
#     'train': '#C04848', 'cv': '#1D9E75',
#     'val':   '#E8A020', 'test': '#1B2B5E',
# }
# CLUSTER_PALETTE = ['#1B2B5E','#1D9E75','#E8A020','#C04848','#7B3F9E','#0E7B8C']

# # ── Figure 1: Model performance ───────────────────────────────────────────────
# fig1 = plt.figure(figsize=(22, 18))
# fig1.suptitle('RBC Conversion Model — Performance (Leakage-Fixed)',
#               fontsize=15, fontweight='bold', y=0.98)
# gs1 = gridspec.GridSpec(3, 3, figure=fig1, hspace=0.42, wspace=0.35)

# ax = fig1.add_subplot(gs1[0, 0])
# for label, y_true, y_prob, c in [
#     ('Train', y_train, y_prob_train, COLORS['train']),
#     ('Validation', y_val, y_prob_val, COLORS['val']),
#     ('Test', y_test, y_prob_test, COLORS['test']),
# ]:
#     fpr, tpr, _ = roc_curve(y_true, y_prob)
#     ax.plot(fpr, tpr, color=c, linewidth=2,
#             label=f'{label} ({roc_auc_score(y_true, y_prob):.3f})')
# ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.4)
# ax.set_xlabel('False Positive Rate', fontsize=10)
# ax.set_ylabel('True Positive Rate', fontsize=10)
# ax.set_title('ROC Curves', fontsize=11, fontweight='bold')
# ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[0, 1])
# for label, y_true, y_prob, c in [
#     ('Train', y_train, y_prob_train, COLORS['train']),
#     ('Validation', y_val, y_prob_val, COLORS['val']),
#     ('Test', y_test, y_prob_test, COLORS['test']),
# ]:
#     prec, rec, _ = precision_recall_curve(y_true, y_prob)
#     ax.plot(rec, prec, color=c, linewidth=2,
#             label=f'{label} ({average_precision_score(y_true, y_prob):.3f})')
# ax.axhline(y.mean(), color='gray', linestyle='--', lw=0.8, label='Baseline')
# ax.set_xlabel('Recall', fontsize=10); ax.set_ylabel('Precision', fontsize=10)
# ax.set_title('Precision-Recall Curves', fontsize=11, fontweight='bold')
# ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[0, 2])
# split_labels = ['Train','CV mean','Validation','Test']
# auc_vals     = [train_auc, cv_auc.mean(), val_auc, test_auc]
# bar_colors   = [COLORS['train'], COLORS['cv'], COLORS['val'], COLORS['test']]
# bars = ax.bar(split_labels, auc_vals, color=bar_colors, edgecolor='white', width=0.55)
# for b, v in zip(bars, auc_vals):
#     ax.text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}',
#             ha='center', va='bottom', fontsize=10, fontweight='bold')
# ax.errorbar(1, cv_auc.mean(), yerr=cv_auc.std()*2,
#             fmt='none', color='black', capsize=5)
# ax.set_ylim(0.5, 1.05); ax.set_ylabel('ROC-AUC', fontsize=10)
# ax.set_title('AUC by Split (CV ±2 SD)', fontsize=11, fontweight='bold')
# ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[1, 0])
# apr_vals = [train_apr, cv_apr.mean(), val_apr, test_apr]
# bars4 = ax.bar(split_labels, apr_vals, color=bar_colors, edgecolor='white', width=0.55)
# for b, v in zip(bars4, apr_vals):
#     ax.text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}',
#             ha='center', va='bottom', fontsize=10, fontweight='bold')
# ax.errorbar(1, cv_apr.mean(), yerr=cv_apr.std()*2,
#             fmt='none', color='black', capsize=5)
# ax.set_ylim(0, 1.05); ax.set_ylabel('Avg Precision', fontsize=10)
# ax.set_title('Avg Precision by Split', fontsize=11, fontweight='bold')
# ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[1, 1])
# fold_x = np.arange(5); w = 0.35
# ax.bar(fold_x-w/2, cv_auc, width=w, color=COLORS['cv'],
#        label='ROC-AUC', edgecolor='white')
# ax.bar(fold_x+w/2, cv_apr, width=w, color=COLORS['train'],
#        label='Avg Precision', edgecolor='white', alpha=0.8)
# ax.axhline(cv_auc.mean(), color=COLORS['cv'],    linestyle='--', lw=1, alpha=0.7)
# ax.axhline(cv_apr.mean(), color=COLORS['train'], linestyle='--', lw=1, alpha=0.7)
# ax.set_xticks(fold_x)
# ax.set_xticklabels([f'Fold {i+1}' for i in range(5)], fontsize=9)
# ax.set_ylim(0.4, 1.0)
# ax.set_title('CV Fold Breakdown', fontsize=11, fontweight='bold')
# ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[1, 2])
# for label, y_prob, c in [
#     ('Train', y_prob_train, COLORS['train']),
#     ('Validation', y_prob_val, COLORS['val']),
#     ('Test', y_prob_test, COLORS['test']),
# ]:
#     ax.hist(y_prob, bins=40, alpha=0.5, color=c, label=label, density=True)
# ax.axvline(best_thresh, color='black', linestyle='--', lw=1.2,
#            label=f'Threshold ({best_thresh:.2f})')
# ax.set_xlabel('Predicted Probability', fontsize=10)
# ax.set_ylabel('Density', fontsize=10)
# ax.set_title('Score Distributions', fontsize=11, fontweight='bold')
# ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[2, 0])
# sorted_idx   = np.argsort(-y_prob_test)
# sorted_y     = np.array(y_test)[sorted_idx]
# n_pos        = sorted_y.sum()
# cum_pos      = np.cumsum(sorted_y)
# pct_contacts = np.arange(1, len(sorted_y)+1) / len(sorted_y) * 100
# pct_captured = cum_pos / n_pos * 100
# ax.plot(pct_contacts, pct_captured, color=COLORS['test'], lw=2, label='Model')
# ax.plot([0,100],[0,100],'k--',lw=0.8,alpha=0.4,label='Random')
# ax.fill_between(pct_contacts, pct_contacts, pct_captured,
#                 alpha=0.08, color=COLORS['test'])
# idx_20 = np.searchsorted(pct_contacts, 20)
# ax.annotate(f"Top 20% → {pct_captured[idx_20]:.0f}% captured",
#             xy=(20, pct_captured[idx_20]),
#             xytext=(35, pct_captured[idx_20]-15),
#             arrowprops=dict(arrowstyle='->', color='black', lw=1), fontsize=9)
# ax.set_xlabel('% parties contacted', fontsize=10)
# ax.set_ylabel('% converters captured', fontsize=10)
# ax.set_title('Cumulative Gain', fontsize=11, fontweight='bold')
# ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[2, 1])
# base_rate = n_pos / len(sorted_y)
# lift_vals  = (cum_pos / np.arange(1, len(sorted_y)+1)) / base_rate
# ax.plot(pct_contacts, lift_vals, color=COLORS['test'], lw=2)
# ax.axhline(1.0, color='gray', linestyle='--', lw=0.8)
# ax.set_xlabel('% parties contacted', fontsize=10)
# ax.set_ylabel('Lift', fontsize=10)
# ax.set_title('Lift Curve', fontsize=11, fontweight='bold')
# ax.spines[['top','right']].set_visible(False)

# ax = fig1.add_subplot(gs1[2, 2])
# top20  = importance_df.head(20)
# fi_c   = [COLORS['test'] if i<5 else COLORS['cv'] if i<10 else '#A0AEC0'
#           for i in range(len(top20))]
# ax.barh(top20['feature'][::-1], top20['importance'][::-1],
#         color=fi_c[::-1], edgecolor='white')
# ax.set_xlabel('Importance (gain)', fontsize=10)
# ax.set_title('Top 20 Features', fontsize=11, fontweight='bold')
# ax.spines[['top','right']].set_visible(False)
# ax.tick_params(axis='y', labelsize=8)

# plt.tight_layout(rect=[0,0,1,0.97])
# plt.savefig(os.path.join(OUTPUT_PATH, "model_performance_charts.png"),
#             dpi=150, bbox_inches='tight')
# plt.close()
# print("\nPerformance charts saved.")

# # ── Figure 2: Clustering (only if clustering ran) ─────────────────────────────
# if do_clustering:
#     CLUSTER_PALETTE = ['#1B2B5E','#1D9E75','#E8A020','#C04848','#7B3F9E','#0E7B8C']

#     fig2 = plt.figure(figsize=(22, 16))
#     fig2.suptitle('RBC Prospect Segmentation — Role-Based Clusters',
#                   fontsize=15, fontweight='bold', y=0.98)
#     gs2 = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.45, wspace=0.35)

#     axA = fig2.add_subplot(gs2[0, 0])
#     axA.plot(list(K_range), inertias, 'o-', color=COLORS['test'], lw=2, markersize=6)
#     axA.axvline(best_k, color=COLORS['cv'], linestyle='--', lw=1.5,
#                 label=f'Chosen k={best_k}')
#     axA.set_xlabel('k', fontsize=10); axA.set_ylabel('Inertia', fontsize=10)
#     axA.set_title('Elbow Method', fontsize=11, fontweight='bold')
#     axA.legend(fontsize=9); axA.spines[['top','right']].set_visible(False)

#     axB = fig2.add_subplot(gs2[0, 1])
#     for i in range(best_k):
#         m = cluster_labels == i
#         axB.scatter(X_pca[m,0], X_pca[m,1],
#                     c=CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)],
#                     alpha=0.5, s=15, label=name_map.get(i,'')[:22])
#     axB.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)', fontsize=10)
#     axB.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)', fontsize=10)
#     axB.set_title('Cluster Map (PCA)', fontsize=11, fontweight='bold')
#     axB.legend(fontsize=7); axB.spines[['top','right']].set_visible(False)

#     axC = fig2.add_subplot(gs2[0, 2])
#     sc = axC.scatter(X_pca[:,0], X_pca[:,1],
#                      c=conversion_prob[high_prob_mask],
#                      cmap='RdYlGn', alpha=0.5, s=15, vmin=0.5, vmax=1.0)
#     plt.colorbar(sc, ax=axC, label='Conversion probability')
#     axC.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)', fontsize=10)
#     axC.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)', fontsize=10)
#     axC.set_title('PCA — Conversion Prob', fontsize=11, fontweight='bold')
#     axC.spines[['top','right']].set_visible(False)

#     axD = fig2.add_subplot(gs2[1, 0])
#     seg_names = [name_map.get(i,'')[:20] for i in range(best_k)]
#     seg_sizes = [int(cluster_summary.loc[i,'n_parties']) for i in range(best_k)]
#     seg_probs = [cluster_summary.loc[i,'conversion_probability'] for i in range(best_k)]
#     x_pos     = np.arange(best_k)
#     bars_d    = axD.bar(x_pos, seg_sizes,
#                         color=[CLUSTER_PALETTE[i%len(CLUSTER_PALETTE)] for i in range(best_k)],
#                         edgecolor='white', width=0.6)
#     ax_twin   = axD.twinx()
#     ax_twin.plot(x_pos, seg_probs, 'o--', color='black', lw=1.5, markersize=7)
#     ax_twin.set_ylabel('Mean conv. prob', fontsize=9)
#     ax_twin.set_ylim(0.4, 1.0)
#     for b, s in zip(bars_d, seg_sizes):
#         axD.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
#                  f'{s:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
#     axD.set_xticks(x_pos)
#     axD.set_xticklabels(seg_names, rotation=30, ha='right', fontsize=8)
#     axD.set_ylabel('# parties', fontsize=10)
#     axD.set_title('Segment Size & Mean Probability', fontsize=11, fontweight='bold')
#     axD.spines[['top','right']].set_visible(False)

#     # Radar chart
#     radar_features = [f for f in [
#         'has_primary_beneficiary', 'is_joint_owner', 'is_trustee',
#         'has_power_of_attorney', 'has_minor', 'n_distinct_roles'
#     ] if f in cluster_summary.columns]

#     if len(radar_features) >= 4:
#         axE = fig2.add_subplot(gs2[1, 1], polar=True)
#         angles = np.linspace(0, 2*np.pi, len(radar_features), endpoint=False).tolist()
#         angles += angles[:1]
#         norm_df = cluster_summary[radar_features].copy()
#         for col in radar_features:
#             rng = norm_df[col].max() - norm_df[col].min()
#             norm_df[col] = (norm_df[col] - norm_df[col].min()) / (rng + 1e-9)
#         for i in range(best_k):
#             vals = norm_df.loc[i, radar_features].tolist() + [norm_df.loc[i, radar_features[0]]]
#             axE.plot(angles, vals, 'o-', lw=1.5,
#                      color=CLUSTER_PALETTE[i%len(CLUSTER_PALETTE)],
#                      label=name_map.get(i,'')[:20], markersize=4)
#             axE.fill(angles, vals, alpha=0.07,
#                      color=CLUSTER_PALETTE[i%len(CLUSTER_PALETTE)])
#         axE.set_xticks(angles[:-1])
#         axE.set_xticklabels([f.replace('_',' ')[:14] for f in radar_features], fontsize=8)
#         axE.set_title('Role Profiles (normalised)', fontsize=11, fontweight='bold', pad=20)
#         axE.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.4, 1.1))
#         axE.spines['polar'].set_visible(False)

#     axF = fig2.add_subplot(gs2[1, 2])
#     box_data = [
#         cluster_profile_df[cluster_profile_df['cluster']==i]['conversion_probability'].values
#         for i in range(best_k)
#     ]
#     bp = axF.boxplot(box_data, patch_artist=True, notch=False)
#     for patch, color in zip(bp['boxes'], CLUSTER_PALETTE[:best_k]):
#         patch.set_facecolor(color); patch.set_alpha(0.8)
#     for median in bp['medians']:
#         median.set_color('white'); median.set_linewidth(2)
#     axF.set_xticklabels([n[:16] for n in seg_names], rotation=30, ha='right', fontsize=8)
#     axF.set_ylabel('Conversion probability', fontsize=10)
#     axF.set_title('Probability Distribution per Segment', fontsize=11, fontweight='bold')
#     axF.spines[['top','right']].set_visible(False)

#     axG = fig2.add_subplot(gs2[2, 0])
#     tier_order = ['Very High','High','Medium','Low']
#     if 'primary_role' in results.columns:
#         tier_role = (results.groupby(['priority_tier','primary_role'])
#                             .size().unstack(fill_value=0))
#         tier_role_pct = tier_role.div(tier_role.sum(axis=1), axis=0) * 100
#         tier_role_pct = tier_role_pct.reindex(
#             [t for t in tier_order if t in tier_role_pct.index])
#         tier_role_pct.plot(kind='bar', stacked=True, ax=axG,
#                            colormap='tab20', edgecolor='white', lw=0.5)
#         axG.set_ylabel('% of parties', fontsize=10)
#         axG.set_xticklabels(axG.get_xticklabels(), rotation=0, fontsize=10)
#         axG.set_title('Role Mix by Priority Tier', fontsize=11, fontweight='bold')
#         axG.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.6,1.05))
#         axG.spines[['top','right']].set_visible(False)

#     axH = fig2.add_subplot(gs2[2, 1])
#     if 'primary_role' in results.columns:
#         role_prob = (results.groupby('primary_role')['conversion_probability']
#                             .mean().sort_values(ascending=True))
#         bars_h = axH.barh(role_prob.index, role_prob.values,
#                           color=COLORS['test'], edgecolor='white')
#         for b, v in zip(bars_h, role_prob.values):
#             axH.text(v+0.003, b.get_y()+b.get_height()/2,
#                      f'{v:.3f}', va='center', fontsize=9)
#         axH.set_xlabel('Mean conversion probability', fontsize=10)
#         axH.set_title('Avg Prob by Primary Role', fontsize=11, fontweight='bold')
#         axH.spines[['top','right']].set_visible(False)

#     # Heatmap
#     axI = fig2.add_subplot(gs2[2, 2])
#     heat_features = [f for f in [
#         'has_primary_beneficiary', 'is_joint_owner', 'is_trustee',
#         'has_power_of_attorney', 'has_minor', 'has_guardian',
#         'has_trading_authority', 'n_distinct_roles'
#     ] if f in cluster_summary.columns]
#     if heat_features:
#         heat_data = cluster_summary[heat_features].copy()
#         heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-9)
#         im = axI.imshow(heat_norm.values, aspect='auto', cmap='Blues')
#         axI.set_xticks(range(len(heat_features)))
#         axI.set_xticklabels([f.replace('_',' ')[:14] for f in heat_features],
#                             rotation=45, ha='right', fontsize=7)
#         axI.set_yticks(range(best_k))
#         axI.set_yticklabels([n[:22] for n in seg_names], fontsize=8)
#         axI.set_title('Role Feature Heatmap (normalised)', fontsize=11, fontweight='bold')
#         plt.colorbar(im, ax=axI, shrink=0.8)
#         for ri in range(best_k):
#             for ci in range(len(heat_features)):
#                 axI.text(ci, ri, f'{heat_norm.values[ri,ci]:.2f}',
#                          ha='center', va='center', fontsize=6,
#                          color='white' if heat_norm.values[ri,ci] > 0.6 else 'black')

#     plt.tight_layout(rect=[0,0,1,0.97])
#     plt.savefig(os.path.join(OUTPUT_PATH, "model_clustering_charts.png"),
#                 dpi=150, bbox_inches='tight')
#     plt.close()
#     print("Clustering charts saved.")

# # ──────────────────────────────────────────────────────────────────────────────
# # STEP 9 — SAVE OUTPUTS
# # ──────────────────────────────────────────────────────────────────────────────
# results.to_csv(os.path.join(OUTPUT_PATH, "conversion_scores_all.csv"), index=False)

# priority = results[results['priority_tier'].isin(['Very High','High'])].copy()
# if do_clustering:
#     priority = priority.merge(
#         results_high[['PartyId','cluster','segment_name']], on='PartyId', how='left'
#     )
# priority.to_csv(os.path.join(OUTPUT_PATH, "conversion_scores_priority.csv"), index=False)
# results.head(1000).to_csv(os.path.join(OUTPUT_PATH, "conversion_top1000.csv"), index=False)

# joblib.dump(model, os.path.join(OUTPUT_PATH, "conversion_model_xgb.pkl"))
# if do_clustering:
#     joblib.dump(scaler,   os.path.join(OUTPUT_PATH, "cluster_scaler.pkl"))
#     joblib.dump(km_final, os.path.join(OUTPUT_PATH, "cluster_kmeans.pkl"))
#     cluster_summary.reset_index().to_csv(
#         os.path.join(OUTPUT_PATH, "cluster_segment_profiles.csv"), index=False)

# pd.DataFrame({
#     'split':         ['Train','CV_mean','CV_std','Validation','Test'],
#     'roc_auc':       [train_auc, cv_auc.mean(), cv_auc.std(), val_auc, test_auc],
#     'avg_precision': [train_apr, cv_apr.mean(), cv_apr.std(), val_apr, test_apr],
# }).to_csv(os.path.join(OUTPUT_PATH, "model_performance_metrics.csv"), index=False)

# importance_df.to_csv(os.path.join(OUTPUT_PATH, "feature_importance.csv"), index=False)

# print(f"""
# {'='*65}
# FINAL SUMMARY
# {'='*65}
#   Train AUC:      {train_auc:.4f}   Train AP:      {train_apr:.4f}
#   CV AUC:         {cv_auc.mean():.4f} ± {cv_auc.std():.4f}
#   Validation AUC: {val_auc:.4f}   Validation AP: {val_apr:.4f}
#   Test AUC:       {test_auc:.4f}   Test AP:       {test_apr:.4f}
#   Threshold:      {best_thresh:.3f}

#   Non-clients scored: {len(results):,}
#   Very High: {(results['priority_tier']=='Very High').sum():,}
#   High:      {(results['priority_tier']=='High').sum():,}
#   Medium:    {(results['priority_tier']=='Medium').sum():,}
#   Low:       {(results['priority_tier']=='Low').sum():,}

#   Output saved to: {OUTPUT_PATH}
# {'='*65}
# """)

Loading master modeling table...
Loaded: 155,154 rows × 110 columns
Class balance: 30.48% positive (clients)

Safe features used in model: 44

Leakage check — mean of hist columns by is_client:
           hist_total_portfolio_value  hist_avg_portfolio_value  hist_peak_portfolio_value  hist_portfolio_growth_pct  hist_portfolio_growth_6m
is_client                                                                                                                                      
0                               0.000                      0.00                      0.000                       0.00                     0.000
1                          503870.804                 234928.98                 432968.616                     108.49                     9.193
  → If non-client (0) means are all zero and client (1) means are non-zero,
    comment out the hist_* block in SAFE_FEATURES above.

Role columns found: ['has_primary_beneficiary', 'has_contingent_beneficiary', 'has_tod_beneficia

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# RBC CONVERSION MODEL — FULL PIPELINE (Feature Engineering + Model)
# Built from raw source files, leakage-free
#
# COLAB SETUP (run in separate cells before this script):
#   !pip install xgboost --quiet
#   from google.colab import files
#   uploaded = files.upload()   # upload all CSV files individually
#   # OR mount Drive and set DATA_PATH below
#
# All CSVs should be in the same folder. Set DATA_PATH accordingly.
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os, glob, warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, roc_curve,
                              precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH   = '/content'          # folder containing all CSVs
OUTPUT_PATH = '/content/output'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── Utility: read CSV with encoding fallback ───────────────────────────────────
def read_csv(fname, **kw):
    path = os.path.join(DATA_PATH, fname)
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not read {fname}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD ALL SOURCE FILES
# ══════════════════════════════════════════════════════════════════════════════
print("Loading source files...")

party   = read_csv('Party.csv')
acct    = read_csv('BroadridgeAccount.csv')
role    = read_csv('BroadridgeAccountRole.csv')
yodlee  = read_csv('YodleeAccount.csv')
email   = read_csv('IndividualEmailResult.csv')
hpr     = read_csv('HouseholdPartyRelationship.csv')
hh      = read_csv('Household.csv')
pr      = read_csv('PartyRelationship.csv')
psd     = read_csv('PartyServiceDetails.csv')
wmo     = read_csv('PartyWMOandAggregationDetails.csv')
cm      = read_csv('CampaignMember.csv')
contrib = read_csv('BroadridgeAccountContributionHistory.csv')
dist    = read_csv('BroadridgeAccountDistributionHistory.csv')

# Load and concatenate all history files
print("Loading account history files (this takes ~1 min)...")
hist_files = sorted(glob.glob(os.path.join(DATA_PATH,
                    'BroadridgeAccountAdditionalHistory*.csv')))
hist = pd.concat([read_csv(os.path.basename(f)) for f in hist_files],
                 ignore_index=True)
print(f"  History rows loaded: {len(hist):,}")

# ── Normalise key ID columns to string ────────────────────────────────────────
for df, cols in [
    (party,   ['PartyId']),
    (acct,    ['PrimaryOwnerPartyId', 'FinancialAccountId']),
    (role,    ['PartyId', 'AccountId']),
    (yodlee,  ['PrimaryOwnerPartyId']),
    (email,   ['PartyId']),
    (hpr,     ['PartyId', 'HouseholdId']),
    (pr,      ['PartyId', 'RelatedPartyId']),
    (psd,     ['PartyId']),
    (wmo,     ['PartyId']),
    (cm,      ['PartyId']),
    (contrib, ['FinancialAccountId']),
    (dist,    ['FinancialAccountId']),
    (hist,    ['FinancialAccountId']),
]:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

print(f"Parties: {len(party):,}  |  Accounts: {len(acct):,}  |  Roles: {len(role):,}")
print(f"Non-clients (target population): "
      f"{(party['ClientIndicator']=='N').sum():,}")
print(f"Clients (positive class):        "
      f"{(party['ClientIndicator']=='Y').sum():,}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — BUILD ACCOUNT-LEVEL AGGREGATIONS
# These are built from the PRIMARY OWNER's perspective and will be joined
# to non-clients via their role on the account.
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding account-level aggregations...")

# ── 2a. Latest portfolio snapshot per account from history ────────────────────
hist['DeltaDate'] = pd.to_datetime(hist['DeltaDate'], errors='coerce')
hist_latest = (hist.sort_values('DeltaDate')
                   .groupby('FinancialAccountId')
                   .last()
                   .reset_index()
                   [['FinancialAccountId','DeltaDate','PortfolioValue',
                     'EquityPercentage','MarginBalance','CashBalance',
                     'FederalCall','HouseCall','YTDMarginInterest']])
hist_latest.columns = ['FinancialAccountId','last_history_date',
                        'acct_portfolio_value','acct_equity_pct',
                        'acct_margin_balance','acct_cash_balance',
                        'acct_federal_call','acct_house_call',
                        'acct_ytd_margin_interest']

# Peak portfolio value
hist_peak = (hist.groupby('FinancialAccountId')['PortfolioValue']
                 .max().reset_index()
                 .rename(columns={'PortfolioValue':'acct_peak_portfolio_value'}))

# ── 2b. Contribution and distribution history per account ─────────────────────
contrib_agg = contrib.groupby('FinancialAccountId').agg(
    acct_cytd_direct_rollover   = ('CYTDDirectRollover',  'sum'),
    acct_cytd_ira_contribution  = ('CYTDIRAContribution',  'sum'),
    acct_cytd_employee_contrib  = ('CYTDEmployeeContribution', 'sum'),
    acct_has_rollover           = ('CYTDDirectRollover',
                                   lambda x: int((x > 0).any())),
).reset_index()

dist_agg = dist.groupby('FinancialAccountId').agg(
    acct_cytd_gross_distribution = ('CYTDGrossDistribution', 'sum'),
    acct_pytd_gross_distribution = ('PYTDGrossDistribution', 'sum'),
    acct_has_distribution        = ('CYTDGrossDistribution',
                                    lambda x: int((x > 0).any())),
).reset_index()

# ── 2c. Merge all account-level features onto BroadridgeAccount ───────────────
acct_feat = (acct
    .merge(hist_latest, on='FinancialAccountId', how='left')
    .merge(hist_peak,   on='FinancialAccountId', how='left')
    .merge(contrib_agg, on='FinancialAccountId', how='left')
    .merge(dist_agg,    on='FinancialAccountId', how='left')
)

# Parse open date → account age in years
acct_feat['OpenDate'] = pd.to_datetime(acct_feat['OpenDate'], errors='coerce')
SNAPSHOT_DATE = pd.Timestamp('2026-02-27')
acct_feat['acct_age_years'] = (
    (SNAPSHOT_DATE - acct_feat['OpenDate']).dt.days / 365.25
).clip(lower=0)

# Advisory account flag
acct_feat['acct_is_advisory'] = (
    acct_feat['AdvisoryAccount'].astype(str).str.lower() == 'true'
).astype(int)

# RMD flag (has required minimum distribution setup)
acct_feat['acct_has_rmd'] = acct_feat['RequiredMinimumDistribution'].notna().astype(int)

# Account type flags
acct_feat['acct_is_ira']   = acct_feat['AccountSubType'].str.contains(
    'IRA|Roth|Retirement|Sep|Simple|Sarsep|Rollover', case=False, na=False).astype(int)
acct_feat['acct_is_trust'] = acct_feat['AccountSubType'].str.contains(
    'Trust|Grantor|Irrevocable', case=False, na=False).astype(int)
acct_feat['acct_is_joint'] = acct_feat['AccountSubType'].str.contains(
    'Joint|WROS|Tenants', case=False, na=False).astype(int)
acct_feat['acct_is_529']   = acct_feat['AccountSubType'].str.contains(
    '529', case=False, na=False).astype(int)

# Primary owner age (proxy for inheritance proximity)
party_age = party[['PartyId','PartyAge']].rename(
    columns={'PartyId':'PrimaryOwnerPartyId','PartyAge':'primary_owner_age'})
acct_feat = acct_feat.merge(party_age, on='PrimaryOwnerPartyId', how='left')
acct_feat['primary_owner_age'] = pd.to_numeric(
    acct_feat['primary_owner_age'], errors='coerce')

# Inheritance proximity: primary owner is 70+
acct_feat['acct_primary_owner_70plus'] = (
    acct_feat['primary_owner_age'] >= 70).astype(int)
acct_feat['acct_primary_owner_80plus'] = (
    acct_feat['primary_owner_age'] >= 80).astype(int)

print(f"  Account features built: {len(acct_feat):,} accounts")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — BUILD NON-CLIENT FEATURES FROM ASSOCIATED ACCOUNTS
# For each non-client, aggregate across all accounts they're named on.
# This is the core of the leakage-free feature set.
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding non-client features from associated accounts...")

# Join role → account features
role_acct = role.merge(
    acct_feat[[
        'FinancialAccountId',
        'PrimaryOwnerPartyId',
        'acct_portfolio_value', 'acct_peak_portfolio_value',
        'acct_equity_pct', 'acct_cash_balance', 'acct_margin_balance',
        'acct_ytd_margin_interest', 'acct_federal_call', 'acct_house_call',
        'acct_cytd_direct_rollover', 'acct_cytd_ira_contribution',
        'acct_cytd_gross_distribution', 'acct_pytd_gross_distribution',
        'acct_has_rollover', 'acct_has_distribution',
        'acct_age_years', 'acct_is_advisory', 'acct_has_rmd',
        'acct_is_ira', 'acct_is_trust', 'acct_is_joint', 'acct_is_529',
        'acct_primary_owner_age',
        'acct_primary_owner_70plus', 'acct_primary_owner_80plus',
        'AccountStatus',
    ]],
    left_on='AccountId', right_on='FinancialAccountId', how='left'
)

# Keep only open accounts (closed accounts are less actionable)
role_acct = role_acct[role_acct['AccountStatus'].isin(
    ['Account Open', np.nan, None]) | role_acct['AccountStatus'].isna()]

# ── Role type flags ────────────────────────────────────────────────────────────
role_acct['is_beneficiary'] = role_acct['FinancialAccountRole'].isin([
    'Primary Beneficiary', 'Contingent Beneficiary',
    'Transfer on Death (TOD) Bene Primary',
    'Transfer on Death (TOD) Bene Contingent',
    'Designated Beneficiary'
]).astype(int)

role_acct['is_primary_bene']    = (role_acct['FinancialAccountRole'] ==
                                    'Primary Beneficiary').astype(int)
role_acct['is_tod_bene']        = role_acct['FinancialAccountRole'].str.contains(
    'TOD', na=False).astype(int)
role_acct['is_trusted_contact'] = (role_acct['FinancialAccountRole'] ==
                                    'Trusted Contact').astype(int)
role_acct['is_minor_role']      = (role_acct['FinancialAccountRole'] ==
                                    'Minor').astype(int)

# ── Aggregate per non-client party ────────────────────────────────────────────
agg_dict = {
    # Number and type of accounts associated with
    'n_accounts_named_on':          ('AccountId',                'nunique'),
    'n_accounts_as_beneficiary':    ('is_beneficiary',           'sum'),
    'n_accounts_as_primary_bene':   ('is_primary_bene',          'sum'),
    'n_accounts_as_tod_bene':       ('is_tod_bene',              'sum'),
    'n_accounts_as_trusted_contact':('is_trusted_contact',       'sum'),
    'n_accounts_as_minor':          ('is_minor_role',            'sum'),

    # Portfolio value of associated accounts
    'associated_total_portfolio':   ('acct_portfolio_value',     'sum'),
    'associated_max_portfolio':     ('acct_portfolio_value',     'max'),
    'associated_mean_portfolio':    ('acct_portfolio_value',     'mean'),
    'associated_peak_portfolio':    ('acct_peak_portfolio_value','max'),

    # Account characteristics
    'n_advisory_accounts':          ('acct_is_advisory',         'sum'),
    'n_ira_accounts':               ('acct_is_ira',              'sum'),
    'n_trust_accounts':             ('acct_is_trust',            'sum'),
    'n_joint_accounts':             ('acct_is_joint',            'sum'),
    'n_529_accounts':               ('acct_is_529',              'sum'),
    'associated_acct_age_max':      ('acct_age_years',           'max'),
    'associated_acct_age_mean':     ('acct_age_years',           'mean'),

    # Inheritance proximity signals
    'any_primary_owner_70plus':     ('acct_primary_owner_70plus','max'),
    'any_primary_owner_80plus':     ('acct_primary_owner_80plus','max'),
    'max_primary_owner_age':        ('acct_primary_owner_age',   'max'),

    # Money movement on associated accounts
    'associated_total_rollover':    ('acct_cytd_direct_rollover','sum'),
    'associated_total_distribution':('acct_cytd_gross_distribution','sum'),
    'associated_has_rollover':      ('acct_has_rollover',        'max'),
    'associated_has_distribution':  ('acct_has_distribution',   'max'),

    # Risk indicators
    'associated_has_rmd':           ('acct_has_rmd',             'max'),
    'associated_margin_balance':    ('acct_margin_balance',      'sum'),
    'associated_any_federal_call':  ('acct_federal_call',        'max'),
}

nc_from_roles = (role_acct.groupby('PartyId')
                           .agg(**agg_dict)
                           .reset_index())

# Flags
nc_from_roles['has_advisory_associated']  = (
    nc_from_roles['n_advisory_accounts'] > 0).astype(int)
nc_from_roles['has_ira_associated']       = (
    nc_from_roles['n_ira_accounts'] > 0).astype(int)
nc_from_roles['has_trust_associated']     = (
    nc_from_roles['n_trust_accounts'] > 0).astype(int)
nc_from_roles['multi_account_named']      = (
    nc_from_roles['n_accounts_named_on'] > 1).astype(int)
nc_from_roles['log_associated_portfolio'] = np.log1p(
    nc_from_roles['associated_total_portfolio'])

print(f"  Non-client role features: {len(nc_from_roles):,} parties × "
      f"{len(nc_from_roles.columns)} columns")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — HOUSEHOLD FEATURES
# Whether the non-client belongs to a client household
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding household features...")

hh_client = hh[hh['HouseholdType'] == 'Client Household'][['HouseholdId']]
hh_client['household_is_client_hh'] = 1

party_hh = hpr.merge(hh_client, on='HouseholdId', how='left')
party_hh['household_is_client_hh'] = party_hh['household_is_client_hh'].fillna(0)
party_hh['in_any_household']        = 1

hh_feats = (party_hh.groupby('PartyId')
                    .agg(
                        household_is_client_hh = ('household_is_client_hh', 'max'),
                        in_any_household        = ('in_any_household',       'max'),
                        n_households            = ('HouseholdId',            'nunique'),
                    ).reset_index())

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — PARTY RELATIONSHIP NETWORK FEATURES
# How many existing clients does this non-client have relationships with?
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding relationship network features...")

client_ids = set(party[party['ClientIndicator'] == 'Y']['PartyId'].astype(str))

pr_nc = pr.copy()
pr_nc['related_is_client'] = pr_nc['RelatedPartyId'].isin(client_ids).astype(int)
pr_nc['party_is_client']   = pr_nc['PartyId'].isin(client_ids).astype(int)

# For non-clients: how many client relationships do they have
rel_feats = (pr_nc[pr_nc['party_is_client'] == 0]
             .groupby('PartyId')
             .agg(
                 n_client_relationships     = ('related_is_client', 'sum'),
                 n_total_relationships      = ('RelatedPartyId',     'nunique'),
             ).reset_index())
rel_feats['pct_relationships_are_clients'] = (
    rel_feats['n_client_relationships'] /
    rel_feats['n_total_relationships'].clip(lower=1)
)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — CAMPAIGN FEATURES
# Has RBC ever targeted or engaged with this non-client in a campaign?
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding campaign engagement features...")

cm['PartyId'] = cm['PartyId'].astype(str)
cm_feats = (cm.groupby('PartyId')
              .agg(
                  n_campaigns_targeted    = ('CampaignId',      'nunique'),
                  ever_responded_campaign = ('HasResponded',     'max'),
                  ever_opted_out_email    = ('HasOptedOutOfEmail','max'),
                  ever_do_not_call        = ('DoNotCall',         'max'),
              ).reset_index())
cm_feats['ever_responded_campaign'] = cm_feats['ever_responded_campaign'].astype(int)
cm_feats['ever_opted_out_email']    = cm_feats['ever_opted_out_email'].astype(int)
cm_feats['ever_do_not_call']        = cm_feats['ever_do_not_call'].astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — PARTY SERVICE DETAILS
# Segment and service level if available
# ══════════════════════════════════════════════════════════════════════════════
psd['PartyId'] = psd['PartyId'].astype(str)
psd_feats = psd[['PartyId','Segment','ServiceLevel']].drop_duplicates('PartyId')
psd_feats['has_segment']       = psd_feats['Segment'].notna().astype(int)
psd_feats['has_service_level'] = psd_feats['ServiceLevel'].notna().astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — PARTY DEMOGRAPHICS (from Party.csv)
# Only age and marital status are populated for non-clients
# ══════════════════════════════════════════════════════════════════════════════
print("\nBuilding demographic features...")

party['PartyAge'] = pd.to_numeric(party['PartyAge'], errors='coerce')
party['age_missing'] = party['PartyAge'].isna().astype(int)
party['PartyAge']    = party['PartyAge'].fillna(party['PartyAge'].median())

# Marital status dummies
marital_dummies = pd.get_dummies(
    party['PartyMaritalStatus'], prefix='marital', dummy_na=False
).astype(int)

# Age bands — useful for inheritance and life-stage signals
party['age_under_40']  = (party['PartyAge'] < 40).astype(int)
party['age_40_to_55']  = ((party['PartyAge'] >= 40) &
                           (party['PartyAge'] < 55)).astype(int)
party['age_55_to_70']  = ((party['PartyAge'] >= 55) &
                           (party['PartyAge'] < 70)).astype(int)
party['age_over_70']   = (party['PartyAge'] >= 70).astype(int)

demo_cols = ['PartyId','ClientIndicator','PartyAge','age_missing',
             'age_under_40','age_40_to_55','age_55_to_70','age_over_70']
demo = pd.concat([party[demo_cols].reset_index(drop=True),
                  marital_dummies.reset_index(drop=True)], axis=1)

# WMO registration
wmo['PartyId'] = wmo['PartyId'].astype(str)
wmo['registered_in_wmo'] = (
    wmo['PartyRegisteredInWMO'].astype(str).str.lower() == 'true').astype(int)
wmo['has_aggregated_accounts'] = (
    wmo['PartyAggregatedAccounts'].astype(str).str.lower() == 'true').astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — ASSEMBLE MASTER TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\nAssembling master feature table...")

master = demo.copy()
master['PartyId'] = master['PartyId'].astype(str)

for feat_df, key in [
    (nc_from_roles, 'PartyId'),
    (hh_feats,      'PartyId'),
    (rel_feats,     'PartyId'),
    (cm_feats,      'PartyId'),
    (psd_feats[['PartyId','has_segment','has_service_level']], 'PartyId'),
    (wmo[['PartyId','registered_in_wmo','has_aggregated_accounts']], 'PartyId'),
]:
    feat_df[key] = feat_df[key].astype(str)
    master = master.merge(feat_df, on=key, how='left')

# Fill numeric NaNs with 0 (party was never named on an account, etc.)
num_cols = master.select_dtypes(include=[np.number]).columns
master[num_cols] = master[num_cols].fillna(0)

# Target: 1 = client, 0 = non-client
master['is_client'] = (master['ClientIndicator'] == 'Y').astype(int)

print(f"\nMaster table: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"  Clients (is_client=1): {master['is_client'].sum():,}")
print(f"  Non-clients (is_client=0): {(master['is_client']==0).sum():,}")

# ── Final leakage check ────────────────────────────────────────────────────────
feature_candidates = [c for c in master.columns
                      if c not in ['PartyId','ClientIndicator','is_client',
                                   'Segment','ServiceLevel']]
check = master.groupby('is_client')[feature_candidates].mean()
ratio = (check.loc[1] / (check.loc[0] + 1e-9)).sort_values(ascending=False)

print("\nTop 15 features by client/non-client ratio (flag if >> 100):")
print(ratio.head(15).round(1).to_string())
print("\nBottom 10 (checking for reverse leakage):")
print(ratio.tail(10).round(3).to_string())

# Save master table
master.to_parquet(os.path.join(OUTPUT_PATH, 'master_modeling_table_v2.parquet'),
                  index=False)
print(f"\nMaster table saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — DEFINE MODEL FEATURES
# ══════════════════════════════════════════════════════════════════════════════
MODEL_FEATURES = [
    # Demographics (observable for both clients and non-clients)
    'PartyAge', 'age_missing',
    'age_under_40', 'age_40_to_55', 'age_55_to_70', 'age_over_70',

    # Associated account characteristics (non-client's view of RBC accounts)
    'n_accounts_named_on',
    'n_accounts_as_beneficiary', 'n_accounts_as_primary_bene',
    'n_accounts_as_tod_bene', 'n_accounts_as_trusted_contact',
    'n_accounts_as_minor',

    # Portfolio value of accounts they're associated with
    'associated_total_portfolio', 'associated_max_portfolio',
    'associated_mean_portfolio', 'associated_peak_portfolio',
    'log_associated_portfolio',

    # Account type signals
    'n_advisory_accounts', 'n_ira_accounts', 'n_trust_accounts',
    'n_joint_accounts', 'n_529_accounts',
    'has_advisory_associated', 'has_ira_associated', 'has_trust_associated',
    'multi_account_named',
    'associated_acct_age_max', 'associated_acct_age_mean',

    # Inheritance proximity
    'any_primary_owner_70plus', 'any_primary_owner_80plus',
    'max_primary_owner_age',

    # Money movement on associated accounts
    'associated_total_rollover', 'associated_total_distribution',
    'associated_has_rollover', 'associated_has_distribution',
    'associated_has_rmd',

    # Household
    'household_is_client_hh', 'in_any_household', 'n_households',

    # Relationship network
    'n_client_relationships', 'n_total_relationships',
    'pct_relationships_are_clients',

    # Campaign engagement
    'n_campaigns_targeted', 'ever_responded_campaign',
    'ever_opted_out_email', 'ever_do_not_call',

    # WMO / digital engagement
    'registered_in_wmo', 'has_aggregated_accounts',

    # Service
    'has_segment', 'has_service_level',
]

# Add marital dummies
marital_feat_cols = [c for c in master.columns if c.startswith('marital_')]
MODEL_FEATURES += marital_feat_cols

# Keep only columns that actually exist
MODEL_FEATURES = [f for f in MODEL_FEATURES if f in master.columns]
print(f"\nModel features: {len(MODEL_FEATURES)}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — TRAIN / VALIDATION / TEST SPLIT (60 / 20 / 20)
# ══════════════════════════════════════════════════════════════════════════════
X         = master[MODEL_FEATURES].copy().fillna(0)
y         = master['is_client'].copy()
party_ids = master['PartyId'].copy()

X_trainval, X_test, y_trainval, y_test, idx_tv, idx_test = train_test_split(
    X, y, party_ids, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_trainval, y_trainval, idx_tv,
    test_size=0.25, random_state=42, stratify=y_trainval)

print(f"\nSplit summary:")
print(f"  Train:      {len(X_train):,} ({y_train.mean():.2%} positive)")
print(f"  Validation: {len(X_val):,}  ({y_val.mean():.2%} positive)")
print(f"  Test:       {len(X_test):,}  ({y_test.mean():.2%} positive)")

# Non-client scoring set
nc_mask    = master['is_client'] == 0
X_score    = X[nc_mask].copy()
meta_score = master[nc_mask][[
    'PartyId', 'PartyAge', 'ClientIndicator',
    'n_accounts_named_on', 'associated_total_portfolio',
    'any_primary_owner_70plus', 'any_primary_owner_80plus',
    'n_advisory_accounts', 'household_is_client_hh',
    'n_client_relationships',
]].copy()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — TRAIN XGBOOST
# ══════════════════════════════════════════════════════════════════════════════
print("\nTraining XGBoost model...")

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"  scale_pos_weight = {spw:.2f}")

model = XGBClassifier(
    n_estimators          = 500,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,
    scale_pos_weight      = spw,
    eval_metric           = 'auc',
    use_label_encoder     = False,
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 20,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"  Best iteration: {model.best_iteration}")

# Cross-validation
print("\nRunning 5-fold CV on train set...")
cv_model = XGBClassifier(
    n_estimators      = model.best_iteration or 200,
    max_depth=5, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=10,
    scale_pos_weight=spw, eval_metric='auc',
    use_label_encoder=False, random_state=42, n_jobs=-1,
)
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(cv_model, X_train, y_train, cv=cv,
                         scoring='roc_auc', n_jobs=-1)
cv_apr = cross_val_score(cv_model, X_train, y_train, cv=cv,
                         scoring='average_precision', n_jobs=-1)
print(f"  CV ROC-AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  CV Avg Prec: {cv_apr.mean():.4f} ± {cv_apr.std():.4f}")

# Evaluate on all splits
y_prob_train = model.predict_proba(X_train)[:, 1]
y_prob_val   = model.predict_proba(X_val)[:, 1]
y_prob_test  = model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_prob_train)
val_auc   = roc_auc_score(y_val,   y_prob_val)
test_auc  = roc_auc_score(y_test,  y_prob_test)
train_apr = average_precision_score(y_train, y_prob_train)
val_apr   = average_precision_score(y_val,   y_prob_val)
test_apr  = average_precision_score(y_test,  y_prob_test)

print(f"\n{'Split':<14} {'ROC-AUC':>10} {'Avg Precision':>15}")
print("─" * 42)
for name, auc, apr in [
    ('Train',      train_auc,     train_apr),
    ('CV mean',    cv_auc.mean(), cv_apr.mean()),
    ('Validation', val_auc,       val_apr),
    ('Test',       test_auc,      test_apr),
]:
    print(f"{name:<14} {auc:>10.4f} {apr:>15.4f}")

if test_auc > 0.95:
    print("\n⚠️  AUC still > 0.95 — check the ratio table printed in Step 9.")
    print("   Any feature with ratio > 100 should be investigated.")
else:
    print("\n✓  AUC in realistic range — model is leakage-free.")

# Threshold
fpr_v, tpr_v, thresholds_v = roc_curve(y_val, y_prob_val)
best_thresh = float(thresholds_v[np.argmax(tpr_v - fpr_v)])
print(f"\nOptimal threshold (Youden's J): {best_thresh:.3f}")
print(classification_report(y_test, (y_prob_test >= best_thresh).astype(int)))

# Feature importance
importance_df = pd.DataFrame({
    'feature':    MODEL_FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features:")
print(importance_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 13 — SCORE NON-CLIENTS + ROLE ENRICHMENT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nScoring {len(X_score):,} non-client parties...")
conversion_prob = model.predict_proba(X_score)[:, 1]

# Primary role for each non-client
ROLE_PRIORITY = [
    'Power of Attorney', 'Trustee', 'Guardian', 'Conservator',
    'Joint Owner', 'Authorized Signer', 'Trading Authority',
    'Trusted Contact',
    'Primary Beneficiary', 'Contingent Beneficiary',
    'Transfer on Death (TOD) Bene Primary',
    'Transfer on Death (TOD) Bene Contingent',
    'Designated Beneficiary', 'Minor',
]

def get_primary_role(party_id):
    roles = role[role['PartyId'] == str(party_id)]['FinancialAccountRole'].tolist()
    for r in ROLE_PRIORITY:
        if r in roles:
            return r
    return roles[0] if roles else 'No role'

def get_all_roles(party_id):
    roles = role[role['PartyId'] == str(party_id)]['FinancialAccountRole'].unique()
    return ', '.join(sorted(set(roles))) if len(roles) else 'None'

# Vectorised role lookup (faster than apply)
role_primary = (role.groupby('PartyId')
                    .apply(lambda g: next(
                        (r for r in ROLE_PRIORITY
                         if r in g['FinancialAccountRole'].values),
                        g['FinancialAccountRole'].iloc[0]
                        if len(g) > 0 else 'No role'
                    ))
                    .reset_index()
                    .rename(columns={0: 'primary_role'}))

role_all = (role.groupby('PartyId')['FinancialAccountRole']
                .apply(lambda x: ', '.join(sorted(set(x))))
                .reset_index()
                .rename(columns={'FinancialAccountRole': 'roles_held'}))

results = meta_score.copy()
results['conversion_probability'] = conversion_prob
results['priority_tier'] = pd.cut(
    results['conversion_probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)
results = results.merge(role_primary, on='PartyId', how='left')
results = results.merge(role_all,     on='PartyId', how='left')
results = results.sort_values('conversion_probability',
                              ascending=False).reset_index(drop=True)
results['rank'] = results.index + 1

print(f"\nPriority tier distribution:")
print(results['priority_tier'].value_counts().sort_index().to_string())
print(f"\nTop 10 targets:")
print(results[['rank','PartyId','conversion_probability','priority_tier',
               'primary_role','associated_total_portfolio',
               'any_primary_owner_70plus']].head(10).to_string(index=False))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 14 — CLUSTERING ON HIGH-PROBABILITY NON-CLIENTS
# ══════════════════════════════════════════════════════════════════════════════
HIGH_PROB_THRESHOLD = 0.50
high_mask = conversion_prob >= HIGH_PROB_THRESHOLD
n_high    = high_mask.sum()
print(f"\nHigh-probability pool (≥ {HIGH_PROB_THRESHOLD}): {n_high:,} parties")

CLUSTER_FEATURES = [f for f in [
    'PartyAge', 'age_under_40', 'age_40_to_55', 'age_55_to_70', 'age_over_70',
    'n_accounts_named_on',
    'n_accounts_as_beneficiary', 'n_accounts_as_primary_bene',
    'n_accounts_as_tod_bene', 'n_accounts_as_trusted_contact',
    'n_accounts_as_minor',
    'associated_total_portfolio', 'log_associated_portfolio',
    'n_advisory_accounts', 'n_ira_accounts', 'n_trust_accounts',
    'any_primary_owner_70plus', 'any_primary_owner_80plus',
    'associated_has_rollover', 'associated_has_distribution',
    'household_is_client_hh', 'n_client_relationships',
    'n_campaigns_targeted', 'ever_responded_campaign',
] if f in master.columns]

X_high    = X_score[high_mask][CLUSTER_FEATURES].fillna(0)
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_high)

max_k    = min(7, max(3, n_high // 100))
K_range  = range(2, max_k + 1)
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    inertias.append(km.fit(X_scaled).inertia_)

if len(inertias) >= 3:
    diffs2 = np.diff(np.diff(inertias))
    best_k = int(list(K_range)[np.argmax(diffs2) + 2])
    best_k = max(3, min(best_k, max_k))
else:
    best_k = max_k

print(f"Optimal k = {best_k}")
km_final       = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_labels = km_final.fit_predict(X_scaled)

# Profile clusters
X_high_profile = X_high.copy()
X_high_profile['cluster']              = cluster_labels
X_high_profile['conversion_probability'] = conversion_prob[high_mask]

cluster_summary = (X_high_profile.groupby('cluster')
                                  .mean().round(3))
cluster_summary['n_parties'] = X_high_profile.groupby('cluster').size()

# Name clusters by z-score of role signals
pop_mean = X_high_profile[CLUSTER_FEATURES].mean()
pop_std  = X_high_profile[CLUSTER_FEATURES].std().replace(0, 1)

ROLE_LABEL_MAP = {
    'n_accounts_as_primary_bene':    'Primary Beneficiaries — Inheritance Ready',
    'n_accounts_as_tod_bene':        'TOD Beneficiaries — Estate Planning Stage',
    'n_accounts_as_trusted_contact': 'Trusted Contacts — Family Network',
    'n_accounts_as_minor':           'Minors — Long-term Growth Opportunity',
    'n_advisory_accounts':           'Advisory Account Adjacent — Upgrade Ready',
    'n_trust_accounts':              'Trust-Associated Parties',
    'any_primary_owner_80plus':      'Near-Inheritance (Owner 80+)',
    'any_primary_owner_70plus':      'Pre-Inheritance (Owner 70+)',
    'household_is_client_hh':        'Client Household Members',
    'n_client_relationships':        'Well-Connected to Client Network',
    'associated_has_rollover':       'Rollover / Transition Opportunity',
    'log_associated_portfolio':      'High-Value Account Associates',
    'n_campaigns_targeted':          'Previously Targeted Prospects',
}

def name_cluster(row):
    z = (row[CLUSTER_FEATURES] - pop_mean) / pop_std
    feat_cols = [c for c in ROLE_LABEL_MAP if c in z.index]
    if not feat_cols:
        return f'Segment {int(row.name)+1}'
    best = z[feat_cols].idxmax()
    if z[best] < 0.25:
        age = row.get('PartyAge', 0)
        if age > 60:
            return 'Older Wealth-Adjacent Prospects (60+)'
        return 'Broad Prospect Pool'
    return ROLE_LABEL_MAP[best]

cluster_summary['segment_name'] = cluster_summary.apply(name_cluster, axis=1)
name_map = cluster_summary['segment_name'].to_dict()

print("\nCluster segments:")
print(f"{'Cluster':<5} {'Segment Name':<45} {'N':>7} {'Avg Prob':>10}")
print("─" * 72)
for i, row in cluster_summary.iterrows():
    print(f"{i:<5} {row['segment_name']:<45} "
          f"{int(row['n_parties']):>7} {row['conversion_probability']:>10.3f}")

# PCA
pca     = PCA(n_components=2, random_state=42)
X_pca   = pca.fit_transform(X_scaled)
pca_var = pca.explained_variance_ratio_

# Attach segments to results
results_high = results[results['priority_tier'].isin(['High','Very High'])].copy()
results_high = results_high.reset_index(drop=True)
results_high['cluster']      = cluster_labels
results_high['segment_name'] = results_high['cluster'].map(name_map)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 15 — CHARTS
# ══════════════════════════════════════════════════════════════════════════════
COLORS = {'train':'#C04848','cv':'#1D9E75','val':'#E8A020','test':'#1B2B5E'}
PALETTE = ['#1B2B5E','#1D9E75','#E8A020','#C04848','#7B3F9E','#0E7B8C','#D4531E']

# ── Figure 1: Model performance ───────────────────────────────────────────────
fig1, axes = plt.subplots(2, 3, figsize=(20, 12))
fig1.suptitle('RBC Conversion Model — Performance Dashboard',
              fontsize=15, fontweight='bold')
axes = axes.flatten()

# ROC
ax = axes[0]
for lbl, yt, yp, c in [
    ('Train', y_train, y_prob_train, COLORS['train']),
    ('Val',   y_val,   y_prob_val,   COLORS['val']),
    ('Test',  y_test,  y_prob_test,  COLORS['test']),
]:
    fpr, tpr, _ = roc_curve(yt, yp)
    ax.plot(fpr, tpr, color=c, lw=2,
            label=f'{lbl} ({roc_auc_score(yt,yp):.3f})')
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# PR curve
ax = axes[1]
for lbl, yt, yp, c in [
    ('Train', y_train, y_prob_train, COLORS['train']),
    ('Val',   y_val,   y_prob_val,   COLORS['val']),
    ('Test',  y_test,  y_prob_test,  COLORS['test']),
]:
    pr_, rc_, _ = precision_recall_curve(yt, yp)
    ax.plot(rc_, pr_, color=c, lw=2,
            label=f'{lbl} ({average_precision_score(yt,yp):.3f})')
ax.axhline(y.mean(), color='gray', ls='--', lw=0.8)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# AUC comparison
ax = axes[2]
lbls = ['Train','CV','Val','Test']
aucs = [train_auc, cv_auc.mean(), val_auc, test_auc]
cols = [COLORS['train'], COLORS['cv'], COLORS['val'], COLORS['test']]
bars = ax.bar(lbls, aucs, color=cols, edgecolor='white', width=0.55)
for b, v in zip(bars, aucs):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.errorbar(1, cv_auc.mean(), yerr=cv_auc.std()*2,
            fmt='none', color='black', capsize=5)
ax.set_ylim(0.4, 1.05); ax.set_ylabel('ROC-AUC')
ax.set_title('AUC by Split (CV ±2SD)', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# Score distribution
ax = axes[3]
for lbl, yp, c in [
    ('Train', y_prob_train, COLORS['train']),
    ('Val',   y_prob_val,   COLORS['val']),
    ('Test',  y_prob_test,  COLORS['test']),
]:
    ax.hist(yp, bins=40, alpha=0.5, color=c, label=lbl, density=True)
ax.axvline(best_thresh, color='black', ls='--', lw=1.2,
           label=f'Threshold ({best_thresh:.2f})')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Density')
ax.set_title('Score Distributions', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# Cumulative gain
ax = axes[4]
sorted_idx   = np.argsort(-y_prob_test)
sorted_y     = np.array(y_test)[sorted_idx]
n_pos        = sorted_y.sum()
cum_pos      = np.cumsum(sorted_y)
pct_c        = np.arange(1, len(sorted_y)+1)/len(sorted_y)*100
pct_cap      = cum_pos/n_pos*100
ax.plot(pct_c, pct_cap, color=COLORS['test'], lw=2, label='Model')
ax.plot([0,100],[0,100],'k--',lw=0.8,alpha=0.4,label='Random')
ax.fill_between(pct_c, pct_c, pct_cap, alpha=0.08, color=COLORS['test'])
idx20 = np.searchsorted(pct_c, 20)
ax.annotate(f"Top 20% → {pct_cap[idx20]:.0f}% captured",
            xy=(20, pct_cap[idx20]), xytext=(35, pct_cap[idx20]-12),
            arrowprops=dict(arrowstyle='->', lw=1), fontsize=9)
ax.set_xlabel('% parties contacted'); ax.set_ylabel('% converters captured')
ax.set_title('Cumulative Gain (Test)', fontweight='bold')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

# Feature importance
ax = axes[5]
top20 = importance_df.head(20)
fi_c  = [COLORS['test'] if i<5 else COLORS['cv'] if i<10 else '#A0AEC0'
         for i in range(len(top20))]
ax.barh(top20['feature'][::-1], top20['importance'][::-1],
        color=fi_c[::-1], edgecolor='white')
ax.set_xlabel('Importance (gain)')
ax.set_title('Top 20 Features', fontweight='bold')
ax.tick_params(axis='y', labelsize=8)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'model_performance_charts.png'),
            dpi=150, bbox_inches='tight')
plt.close()

# ── Figure 2: Clustering ──────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(2, 3, figsize=(20, 12))
fig2.suptitle('RBC Prospect Segmentation — Role-Based Clusters',
              fontsize=15, fontweight='bold')
axes2 = axes2.flatten()

# Elbow
ax = axes2[0]
ax.plot(list(K_range), inertias, 'o-', color=COLORS['test'], lw=2, markersize=6)
ax.axvline(best_k, color=COLORS['cv'], ls='--', lw=1.5, label=f'k={best_k}')
ax.set_xlabel('k'); ax.set_ylabel('Inertia')
ax.set_title('Elbow Method', fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)

# PCA scatter by cluster
ax = axes2[1]
for i in range(best_k):
    m = cluster_labels == i
    ax.scatter(X_pca[m,0], X_pca[m,1], c=PALETTE[i%len(PALETTE)],
               alpha=0.5, s=15, label=name_map.get(i,'')[:25])
ax.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)')
ax.set_title('Cluster Map (PCA)', fontweight='bold')
ax.legend(fontsize=7); ax.spines[['top','right']].set_visible(False)

# PCA coloured by probability
ax = axes2[2]
sc = ax.scatter(X_pca[:,0], X_pca[:,1],
                c=conversion_prob[high_mask], cmap='RdYlGn',
                alpha=0.5, s=15, vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=ax, label='Conv. probability')
ax.set_xlabel(f'PC1 ({pca_var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_var[1]*100:.1f}%)')
ax.set_title('PCA — Conv. Probability', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# Segment size + mean probability
ax = axes2[3]
seg_names = [name_map.get(i,'')[:22] for i in range(best_k)]
seg_sizes = [int(cluster_summary.loc[i,'n_parties']) for i in range(best_k)]
seg_probs = [cluster_summary.loc[i,'conversion_probability'] for i in range(best_k)]
xpos = np.arange(best_k)
bars_d = ax.bar(xpos, seg_sizes,
                color=[PALETTE[i%len(PALETTE)] for i in range(best_k)],
                edgecolor='white', width=0.6)
ax2t = ax.twinx()
ax2t.plot(xpos, seg_probs, 'o--', color='black', lw=1.5, markersize=7)
ax2t.set_ylabel('Mean conv. prob', fontsize=9); ax2t.set_ylim(0.4, 1.0)
for b, s in zip(bars_d, seg_sizes):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{s:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(xpos)
ax.set_xticklabels(seg_names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('# parties'); ax.set_title('Segment Size & Avg Probability', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# Boxplot of probability per cluster
ax = axes2[4]
box_data = [X_high_profile[X_high_profile['cluster']==i]
            ['conversion_probability'].values for i in range(best_k)]
bp = ax.boxplot(box_data, patch_artist=True)
for patch, c in zip(bp['boxes'], PALETTE[:best_k]):
    patch.set_facecolor(c); patch.set_alpha(0.8)
for med in bp['medians']:
    med.set_color('white'); med.set_linewidth(2)
ax.set_xticklabels([n[:18] for n in seg_names], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Conversion probability')
ax.set_title('Probability per Segment', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# Role mix by priority tier
ax = axes2[5]
if 'primary_role' in results.columns:
    tier_order = ['Very High','High','Medium','Low']
    tr = (results.groupby(['priority_tier','primary_role'])
                 .size().unstack(fill_value=0))
    tr_pct = tr.div(tr.sum(axis=1), axis=0) * 100
    tr_pct = tr_pct.reindex([t for t in tier_order if t in tr_pct.index])
    tr_pct.plot(kind='bar', stacked=True, ax=ax,
                colormap='tab20', edgecolor='white', lw=0.5)
    ax.set_ylabel('% of parties')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10)
    ax.set_title('Role Mix by Priority Tier', fontweight='bold')
    ax.legend(fontsize=7, bbox_to_anchor=(1.6, 1.0))
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'clustering_charts.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("Charts saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 16 — SAVE ALL OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════
results.to_csv(os.path.join(OUTPUT_PATH, 'conversion_scores_all.csv'),
               index=False)

priority = results[results['priority_tier'].isin(['Very High','High'])].copy()
priority = priority.merge(
    results_high[['PartyId','cluster','segment_name']], on='PartyId', how='left')
priority.to_csv(os.path.join(OUTPUT_PATH, 'conversion_scores_priority.csv'),
                index=False)

results.head(1000).to_csv(os.path.join(OUTPUT_PATH, 'conversion_top1000.csv'),
                           index=False)

cluster_summary.reset_index().to_csv(
    os.path.join(OUTPUT_PATH, 'cluster_profiles.csv'), index=False)

joblib.dump(model,    os.path.join(OUTPUT_PATH, 'conversion_model_xgb.pkl'))
joblib.dump(scaler,   os.path.join(OUTPUT_PATH, 'cluster_scaler.pkl'))
joblib.dump(km_final, os.path.join(OUTPUT_PATH, 'cluster_kmeans.pkl'))

importance_df.to_csv(os.path.join(OUTPUT_PATH, 'feature_importance.csv'),
                     index=False)

pd.DataFrame({
    'split':         ['Train','CV_mean','CV_std','Validation','Test'],
    'roc_auc':       [train_auc, cv_auc.mean(), cv_auc.std(), val_auc, test_auc],
    'avg_precision': [train_apr, cv_apr.mean(), cv_apr.std(), val_apr, test_apr],
}).to_csv(os.path.join(OUTPUT_PATH, 'model_metrics.csv'), index=False)

# Download all outputs in Colab
try:
    from google.colab import files
    for fname in ['conversion_scores_all.csv','conversion_scores_priority.csv',
                  'conversion_top1000.csv','cluster_profiles.csv',
                  'model_performance_charts.png','clustering_charts.png',
                  'feature_importance.csv','model_metrics.csv']:
        files.download(os.path.join(OUTPUT_PATH, fname))
except ImportError:
    pass  # not in Colab

print(f"""
{'='*65}
COMPLETE
{'='*65}
  Test AUC:  {test_auc:.4f}   Test AP:  {test_apr:.4f}
  CV AUC:    {cv_auc.mean():.4f} ± {cv_auc.std():.4f}

  Non-clients scored: {len(results):,}
    Very High: {(results['priority_tier']=='Very High').sum():,}
    High:      {(results['priority_tier']=='High').sum():,}
    Medium:    {(results['priority_tier']=='Medium').sum():,}
    Low:       {(results['priority_tier']=='Low').sum():,}

  Clusters (k={best_k}):
""" + "\n".join(
    f"    {i}: {row['segment_name']} — {int(row['n_parties']):,} parties"
    for i, row in cluster_summary.iterrows()
) + f"""

  Outputs: {OUTPUT_PATH}
{'='*65}
""")

Loading source files...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Party.csv'

In [ ]:
remaining_features = [
    'PartyAge', 'age_missing', 'networth_encoded', 'income_encoded',
    'total_accounts', 'open_accounts', 'tenure_years', 'account_type_diversity',
    'has_external_account', 'n_external_accounts', 'total_external_balance',
    'has_external_401k', 'has_external_ira', 'has_held_away',
    'total_cytd_inflow', 'total_cytd_outflow', 'has_rollover', 'has_large_withdrawal',
    'emails_sent', 'emails_opened', 'open_rate', 'click_rate', 'is_engager', 'is_clicker',
]
existing = [f for f in remaining_features if f in master.columns]

print("=== Mean by is_client for all remaining features ===")
comparison = master.groupby('is_client')[existing].mean().round(4).T
comparison.columns = ['non_client', 'client']
comparison['ratio'] = (comparison['client'] / (comparison['non_client'] + 1e-9)).round(1)
comparison['non_client_zeros_pct'] = (
    (master[master['is_client']==0][existing] == 0).mean() * 100
).round(1)
print(comparison.sort_values('ratio', ascending=False).to_string())

=== Mean by is_client for all remaining features ===
                        non_client      client         ratio  non_client_zeros_pct
total_cytd_outflow          0.0000  42800.4394  4.280044e+13                 100.0
total_external_balance      0.0000  23709.3598  2.370936e+13                 100.0
total_cytd_inflow           0.0000  19709.9170  1.970992e+13                 100.0
tenure_years                0.0000     10.8190  1.081900e+10                 100.0
emails_sent                 0.0000      2.7131  2.713100e+09                 100.0
total_accounts              0.0000      1.7691  1.769100e+09                 100.0
emails_opened               0.0000      1.6604  1.660400e+09                 100.0
open_accounts               0.0000      1.1896  1.189600e+09                 100.0
account_type_diversity      0.0000      0.9075  9.075000e+08                 100.0
is_engager                  0.0000      0.2784  2.784000e+08                 100.0
open_rate                   0.0000

In [ ]:
# Understand what ClientIndicator values actually exist
print("=== ClientIndicator distribution ===")
print(master['ClientIndicator'].value_counts().to_string())
print()

# Check if there are any non-clients with non-zero account activity
print("=== Non-clients with at least one non-zero feature ===")
non_clients = master[master['is_client'] == 0].copy()
feature_cols = [c for c in master.columns if c not in
                ['is_client','PartyId','ClientIndicator','PartyAge','age_missing']]
non_clients['any_nonzero'] = (non_clients[feature_cols] != 0).any(axis=1)
print(non_clients['any_nonzero'].value_counts())
print()

# What does ClientIndicator look like — is there a sub-type of non-client
# that has actual account activity?
print("=== Mean PartyAge and feature count by ClientIndicator ===")
master['n_nonzero_features'] = (master[feature_cols] != 0).sum(axis=1)
print(master.groupby('ClientIndicator')[['PartyAge','n_nonzero_features',
      'total_accounts','tenure_years']].mean().round(2).to_string())
print()

# Check marital status columns — do non-clients have any of these?
marital_cols = [c for c in master.columns if c.startswith('marital_')]
if marital_cols:
    print("=== Marital columns mean by is_client ===")
    print(master.groupby('is_client')[marital_cols].mean().round(3).T.to_string())

=== ClientIndicator distribution ===
ClientIndicator
N    107863
Y     47284

=== Non-clients with at least one non-zero feature ===
any_nonzero
True    107870
Name: count, dtype: int64

=== Mean PartyAge and feature count by ClientIndicator ===
                 PartyAge  n_nonzero_features  total_accounts  tenure_years
ClientIndicator                                                            
N                   51.58               49.36            0.00          0.00
Y                   62.67               58.20            1.77         10.82

=== Marital columns mean by is_client ===
is_client                     0      1
marital_Divorced          0.007  0.035
marital_Domestic Partner  0.000  0.002
marital_Married           0.186  0.502
marital_Single            0.746  0.214
marital_Widowed           0.008  0.061
